Phase 2 — Data Preprocessing
Phase 2 objective

The goal of this phase is to transform the verified CIC-IDS2018 raw flow data into a clean, consistent, chronologically ordered dataset suitable for the next phases.

Our Phase 2 pipeline is:

Raw CIC-IDS2018 CSVs

        ↓
Metadata & provenance

        ↓
Remove malformed/header-like rows

        ↓
Timestamp conversion

        ↓
Invalid timestamp handling

        ↓
Numeric type normalization

        ↓
∞ / -∞ handling

        ↓
Missing-value handling

        ↓
Duplicate handling

        ↓
Chronological ordering

        ↓
Label normalization

        ↓
Processed dataset


2.1–2.4 — Core Data Preprocessing

Yes. Let's keep 2.1 to 2.4 in one code cell so you don't have to keep running small cells.

These four steps will be:

2.1 Dataset Metadata & Provenance

2.2 Timestamp Conversion

2.3 Remove malformed/header-like rows

2.4 Chronological Ordering

We will not handle missing values, Inf, duplicates, SMOTE, or feature engineering yet.

The code processes the 10 files in chunks so we don't try to load the entire 6.9 GB dataset into memory at once.

# Phase 2 — Data Preprocessing

## 2.0 — Load CIC-IDS2018 Dataset

In this phase, the verified CIC-IDS2018 dataset stored in the project's private Hugging Face repository is accessed from Google Colab.

The original dataset files are kept unchanged. They will be used as the input for the preprocessing pipeline.

In [5]:
# ============================================================
# PHASE 2 — DATA PREPROCESSING
# 2.0 — Load CIC-IDS2018 Dataset
# ============================================================

# Install the Hugging Face Hub library
!pip -q install -U huggingface_hub

# Import required functions
from huggingface_hub import login, list_repo_files, hf_hub_download

# ------------------------------------------------------------
# 1. Authenticate with Hugging Face
# ------------------------------------------------------------
login()

# ------------------------------------------------------------
# 2. Project dataset repository
# ------------------------------------------------------------
REPO_ID = "PrakulShetty/NetVision-AI-SIH26153"
REPO_TYPE = "dataset"

from huggingface_hub import whoami

user = whoami()

print("Hugging Face authentication successful!")
print("Logged-in user:", user["name"])

Hugging Face authentication successful!
Logged-in user: PrakulShetty


In [7]:
from huggingface_hub import hf_hub_download

REPO_ID = "PrakulShetty/NetVision-AI-SIH26153"

FILE_NAME = "Friday-16-02-2018_TrafficForML_CICFlowMeter.csv"

file_path = hf_hub_download(
    repo_id=REPO_ID,
    filename=FILE_NAME,
    repo_type="dataset"
)

print("SUCCESS")
print("File:", FILE_NAME)
print("Local path:", file_path)

Friday-16-02-2018_TrafficForML_CICFlowMe(…): reconstructing file:   0%|          |  0.00B /  334MB            

Friday-16-02-2018_TrafficForML_CICFlowMe(…): downloading bytes:           |  0.00B            

SUCCESS
File: Friday-16-02-2018_TrafficForML_CICFlowMeter.csv
Local path: /root/.cache/huggingface/hub/datasets--PrakulShetty--NetVision-AI-SIH26153/snapshots/02a63cdb2b74b1eb6a30ceb6d7a77e97c9ca09b8/Friday-16-02-2018_TrafficForML_CICFlowMeter.csv


In [8]:
# ============================================================
# PHASE 2 — DATA PREPROCESSING
# 2.0 — Verify Access to All CIC-IDS2018 Files
# ============================================================

from huggingface_hub import hf_hub_download

REPO_ID = "PrakulShetty/NetVision-AI-SIH26153"
REPO_TYPE = "dataset"

CSV_FILES = [
    "Friday-02-03-2018_TrafficForML_CICFlowMeter.csv",
    "Friday-16-02-2018_TrafficForML_CICFlowMeter.csv",
    "Friday-23-02-2018_TrafficForML_CICFlowMeter.csv",
    "Thuesday-20-02-2018_TrafficForML_CICFlowMeter.csv",
    "Thursday-01-03-2018_TrafficForML_CICFlowMeter.csv",
    "Thursday-15-02-2018_TrafficForML_CICFlowMeter.csv",
    "Thursday-22-02-2018_TrafficForML_CICFlowMeter.csv",
    "Wednesday-14-02-2018_TrafficForML_CICFlowMeter.csv",
    "Wednesday-21-02-2018_TrafficForML_CICFlowMeter.csv",
    "Wednesday-28-02-2018_TrafficForML_CICFlowMeter.csv",
]

dataset_paths = {}

print("=" * 70)
print("VERIFYING CIC-IDS2018 FILE ACCESS")
print("=" * 70)

for i, file_name in enumerate(CSV_FILES, start=1):

    print(f"\n[{i}/10] Accessing: {file_name}")

    path = hf_hub_download(
        repo_id=REPO_ID,
        filename=file_name,
        repo_type=REPO_TYPE
    )

    dataset_paths[file_name] = path

    print("      ✓ Accessible")

print("\n" + "=" * 70)
print("DATASET ACCESS VERIFICATION COMPLETED")
print("=" * 70)

print(f"\nSuccessfully accessed: {len(dataset_paths)}/10 files")

VERIFYING CIC-IDS2018 FILE ACCESS

[1/10] Accessing: Friday-02-03-2018_TrafficForML_CICFlowMeter.csv


Friday-02-03-2018_TrafficForML_CICFlowMe(…): reconstructing file:   0%|          |  0.00B /  352MB            

Friday-02-03-2018_TrafficForML_CICFlowMe(…): downloading bytes:           |  0.00B            

      ✓ Accessible

[2/10] Accessing: Friday-16-02-2018_TrafficForML_CICFlowMeter.csv
      ✓ Accessible

[3/10] Accessing: Friday-23-02-2018_TrafficForML_CICFlowMeter.csv


Friday-23-02-2018_TrafficForML_CICFlowMe(…): reconstructing file:   0%|          |  0.00B /  383MB            

Friday-23-02-2018_TrafficForML_CICFlowMe(…): downloading bytes:           |  0.00B            

... (18 outputs left out)

      ✓ Accessible

[10/10] Accessing: Wednesday-28-02-2018_TrafficForML_CICFlowMeter.csv


Wednesday-28-02-2018_TrafficForML_CICFlo(…): reconstructing file:   0%|          |  0.00B /  209MB            

Wednesday-28-02-2018_TrafficForML_CICFlo(…): downloading bytes:           |  0.00B            

      ✓ Accessible

DATASET ACCESS VERIFICATION COMPLETED

Successfully accessed: 10/10 files


## 2.1 — Dataset Metadata & Provenance

Before preprocessing the CIC-IDS2018 files, basic metadata is recorded for each source file. This provides a reproducible reference for the dataset used in the preprocessing pipeline.

The metadata includes file dimensions, schema information, timestamp and label availability, and the Hugging Face source repository.

The original dataset files are not modified during this step.

In [9]:
# ============================================================
# PHASE 2 — DATA PREPROCESSING
# 2.1 — Dataset Metadata & Provenance
# ============================================================

import os
import json
import pandas as pd

# ------------------------------------------------------------
# Source information
# ------------------------------------------------------------
REPO_ID = "PrakulShetty/NetVision-AI-SIH26153"
REPO_TYPE = "dataset"

# Metadata will be stored here
metadata_records = []

print("=" * 70)
print("PHASE 2.1 — DATASET METADATA & PROVENANCE")
print("=" * 70)

# ------------------------------------------------------------
# Process each already-cached dataset file
# ------------------------------------------------------------
for i, (file_name, file_path) in enumerate(dataset_paths.items(), start=1):

    print(f"\n[{i}/{len(dataset_paths)}] Inspecting: {file_name}")

    # Read only the header first
    header_df = pd.read_csv(
        file_path,
        nrows=0
    )

    columns = header_df.columns.tolist()

    # --------------------------------------------------------
    # Count rows without loading the complete file into memory
    # --------------------------------------------------------
    row_count = 0

    for chunk in pd.read_csv(
        file_path,
        chunksize=100_000,
        usecols=[columns[0]]
    ):
        row_count += len(chunk)

    # --------------------------------------------------------
    # Read a small sample for datatype information
    # --------------------------------------------------------
    sample_df = pd.read_csv(
        file_path,
        nrows=1000,
        low_memory=False
    )

    # --------------------------------------------------------
    # Record metadata
    # --------------------------------------------------------
    record = {
        "source_repository": REPO_ID,
        "repository_type": REPO_TYPE,
        "file_name": file_name,
        "number_of_rows": row_count,
        "number_of_columns": len(columns),
        "has_timestamp": "Timestamp" in columns,
        "has_label": "Label" in columns,
        "columns": columns,
        "sample_dtypes": {
            column: str(dtype)
            for column, dtype
            in sample_df.dtypes.items()
        }
    }

    metadata_records.append(record)

    print(f"  Rows              : {row_count:,}")
    print(f"  Columns           : {len(columns)}")
    print(f"  Timestamp present : {'Timestamp' in columns}")
    print(f"  Label present     : {'Label' in columns}")


# ------------------------------------------------------------
# Save metadata
# ------------------------------------------------------------
metadata_path = "/content/phase2_dataset_metadata.json"

with open(metadata_path, "w") as f:
    json.dump(
        metadata_records,
        f,
        indent=4
    )


# ------------------------------------------------------------
# Create compact summary table
# ------------------------------------------------------------
metadata_summary = pd.DataFrame([
    {
        "File": record["file_name"],
        "Rows": record["number_of_rows"],
        "Columns": record["number_of_columns"],
        "Timestamp": record["has_timestamp"],
        "Label": record["has_label"]
    }
    for record in metadata_records
])

print("\n" + "=" * 70)
print("METADATA SUMMARY")
print("=" * 70)

display(metadata_summary)

print("\nMetadata saved to:")
print(metadata_path)

PHASE 2.1 — DATASET METADATA & PROVENANCE

[1/10] Inspecting: Friday-02-03-2018_TrafficForML_CICFlowMeter.csv
  Rows              : 1,048,575
  Columns           : 80
  Timestamp present : True
  Label present     : True

[2/10] Inspecting: Friday-16-02-2018_TrafficForML_CICFlowMeter.csv


/tmp/ipykernel_5303/1509678882.py:43: DtypeWarning: Columns (0) have mixed types. Specify dtype option on import or set low_memory=False.
  for chunk in pd.read_csv(


  Rows              : 1,048,575
  Columns           : 80
  Timestamp present : True
  Label present     : True

[3/10] Inspecting: Friday-23-02-2018_TrafficForML_CICFlowMeter.csv
  Rows              : 1,048,575
  Columns           : 80
  Timestamp present : True
  Label present     : True

[4/10] Inspecting: Thuesday-20-02-2018_TrafficForML_CICFlowMeter.csv
  Rows              : 7,948,748
  Columns           : 84
  Timestamp present : True
  Label present     : True

[5/10] Inspecting: Thursday-01-03-2018_TrafficForML_CICFlowMeter.csv


/tmp/ipykernel_5303/1509678882.py:43: DtypeWarning: Columns (0) have mixed types. Specify dtype option on import or set low_memory=False.
  for chunk in pd.read_csv(
/tmp/ipykernel_5303/1509678882.py:43: DtypeWarning: Columns (0) have mixed types. Specify dtype option on import or set low_memory=False.
  for chunk in pd.read_csv(


  Rows              : 331,125
  Columns           : 80
  Timestamp present : True
  Label present     : True

[6/10] Inspecting: Thursday-15-02-2018_TrafficForML_CICFlowMeter.csv
  Rows              : 1,048,575
  Columns           : 80
  Timestamp present : True
  Label present     : True

[7/10] Inspecting: Thursday-22-02-2018_TrafficForML_CICFlowMeter.csv
  Rows              : 1,048,575
  Columns           : 80
  Timestamp present : True
  Label present     : True

[8/10] Inspecting: Wednesday-14-02-2018_TrafficForML_CICFlowMeter.csv
  Rows              : 1,048,575
  Columns           : 80
  Timestamp present : True
  Label present     : True

[9/10] Inspecting: Wednesday-21-02-2018_TrafficForML_CICFlowMeter.csv
  Rows              : 1,048,575
  Columns           : 80
  Timestamp present : True
  Label present     : True

[10/10] Inspecting: Wednesday-28-02-2018_TrafficForML_CICFlowMeter.csv


/tmp/ipykernel_5303/1509678882.py:43: DtypeWarning: Columns (0) have mixed types. Specify dtype option on import or set low_memory=False.
  for chunk in pd.read_csv(
/tmp/ipykernel_5303/1509678882.py:43: DtypeWarning: Columns (0) have mixed types. Specify dtype option on import or set low_memory=False.
  for chunk in pd.read_csv(
/tmp/ipykernel_5303/1509678882.py:43: DtypeWarning: Columns (0) have mixed types. Specify dtype option on import or set low_memory=False.
  for chunk in pd.read_csv(
/tmp/ipykernel_5303/1509678882.py:43: DtypeWarning: Columns (0) have mixed types. Specify dtype option on import or set low_memory=False.
  for chunk in pd.read_csv(
/tmp/ipykernel_5303/1509678882.py:43: DtypeWarning: Columns (0) have mixed types. Specify dtype option on import or set low_memory=False.
  for chunk in pd.read_csv(
/tmp/ipykernel_5303/1509678882.py:43: DtypeWarning: Columns (0) have mixed types. Specify dtype option on import or set low_memory=False.
  for chunk in pd.read_csv(
/tmp

  Rows              : 613,104
  Columns           : 80
  Timestamp present : True
  Label present     : True

METADATA SUMMARY


,File,Rows,Columns,Timestamp,Label
0,Friday-02-03-2018_TrafficForML_CICFlowMeter.csv,1048575,80,True,True
1,Friday-16-02-2018_TrafficForML_CICFlowMeter.csv,1048575,80,True,True
2,Friday-23-02-2018_TrafficForML_CICFlowMeter.csv,1048575,80,True,True
3,Thuesday-20-02-2018_TrafficForML_CICFlowMeter.csv,7948748,84,True,True
4,Thursday-01-03-2018_TrafficForML_CICFlowMeter.csv,331125,80,True,True
5,Thursday-15-02-2018_TrafficForML_CICFlowMeter.csv,1048575,80,True,True
6,Thursday-22-02-2018_TrafficForML_CICFlowMeter.csv,1048575,80,True,True
7,Wednesday-14-02-2018_TrafficForML_CICFlowMeter...,1048575,80,True,True
8,Wednesday-21-02-2018_TrafficForML_CICFlowMeter...,1048575,80,True,True
9,Wednesday-28-02-2018_TrafficForML_CICFlowMeter...,613104,80,True,True



Metadata saved to:
/content/phase2_dataset_metadata.json


## 2.2 — Timestamp Conversion

The `Timestamp` column is converted from its raw string representation into a proper datetime format.

The CIC-IDS2018 timestamps follow the format:

`DD/MM/YYYY HH:MM:SS`

During this step, timestamps that cannot be parsed are identified. Suspicious but technically parseable timestamps, such as the previously observed 1970 timestamps, are not removed yet and will be investigated separately.

The converted timestamp will later be used to establish chronological ordering and construct the temporal representation required for attack forecasting.

In [10]:
# ============================================================
# PHASE 2 — DATA PREPROCESSING
# 2.2 — Timestamp Conversion & Malformed Row Handling
# ============================================================

import os
import pandas as pd

# ------------------------------------------------------------
# Configuration
# ------------------------------------------------------------

OUTPUT_DIR = "/content/phase2_processed"
os.makedirs(OUTPUT_DIR, exist_ok=True)

CHUNK_SIZE = 100_000

TIMESTAMP_FORMAT = "%d/%m/%Y %H:%M:%S"

print("=" * 70)
print("PHASE 2.2 — TIMESTAMP CONVERSION & MALFORMED ROW HANDLING")
print("=" * 70)

# ------------------------------------------------------------
# Store processing results
# ------------------------------------------------------------

processing_summary = []

# ------------------------------------------------------------
# Process all 10 dataset files
# ------------------------------------------------------------

for file_number, (file_name, input_path) in enumerate(
    dataset_paths.items(), start=1
):

    print(f"\n[{file_number}/10] Processing:")
    print(file_name)

    output_path = os.path.join(
        OUTPUT_DIR,
        file_name
    )

    total_rows = 0
    valid_rows = 0
    malformed_label_rows = 0
    invalid_timestamp_rows = 0

    first_timestamp = None
    last_timestamp = None

    first_chunk = True

    # --------------------------------------------------------
    # Read the file in chunks
    # --------------------------------------------------------

    for chunk in pd.read_csv(
        input_path,
        chunksize=CHUNK_SIZE,
        low_memory=False
    ):

        total_rows += len(chunk)

        # ----------------------------------------------------
        # Identify malformed header-like rows
        # ----------------------------------------------------
        #
        # Phase 1 identified rows where Label itself is
        # "Label". These are repeated header-like rows.
        # ----------------------------------------------------

        malformed_mask = (
            chunk["Label"]
            .astype(str)
            .str.strip()
            .eq("Label")
        )

        malformed_count = int(malformed_mask.sum())
        malformed_label_rows += malformed_count

        # Remove malformed header-like rows
        chunk = chunk.loc[~malformed_mask].copy()

        # ----------------------------------------------------
        # Convert Timestamp to datetime
        # ----------------------------------------------------

        chunk["Timestamp"] = pd.to_datetime(
            chunk["Timestamp"],
            format=TIMESTAMP_FORMAT,
            errors="coerce"
        )

        # ----------------------------------------------------
        # Identify timestamps that could not be converted
        # ----------------------------------------------------

        invalid_mask = chunk["Timestamp"].isna()

        invalid_count = int(invalid_mask.sum())
        invalid_timestamp_rows += invalid_count

        # Remove rows with completely invalid timestamps
        chunk = chunk.loc[~invalid_mask].copy()

        # ----------------------------------------------------
        # Track valid timestamp range
        # ----------------------------------------------------

        if not chunk.empty:

            chunk_min = chunk["Timestamp"].min()
            chunk_max = chunk["Timestamp"].max()

            if first_timestamp is None or chunk_min < first_timestamp:
                first_timestamp = chunk_min

            if last_timestamp is None or chunk_max > last_timestamp:
                last_timestamp = chunk_max

        valid_rows += len(chunk)

        # ----------------------------------------------------
        # Save processed chunk
        # ----------------------------------------------------

        # CSV files cannot preserve pandas datetime dtype.
        # We therefore save the timestamp in an unambiguous
        # ISO-style format for the next preprocessing stages.

        chunk["Timestamp"] = chunk["Timestamp"].dt.strftime(
            "%Y-%m-%d %H:%M:%S"
        )

        chunk.to_csv(
            output_path,
            mode="w" if first_chunk else "a",
            header=first_chunk,
            index=False
        )

        first_chunk = False

    # --------------------------------------------------------
    # Store summary for this file
    # --------------------------------------------------------

    processing_summary.append({
        "File": file_name,
        "Input Rows": total_rows,
        "Valid Rows": valid_rows,
        "Malformed Header Rows Removed": malformed_label_rows,
        "Invalid Timestamp Rows Removed": invalid_timestamp_rows,
        "First Valid Timestamp": str(first_timestamp),
        "Last Valid Timestamp": str(last_timestamp),
        "Output File": output_path
    })

    print(f"  Input rows              : {total_rows:,}")
    print(f"  Valid rows              : {valid_rows:,}")
    print(f"  Malformed header rows   : {malformed_label_rows:,}")
    print(f"  Invalid timestamp rows  : {invalid_timestamp_rows:,}")
    print(f"  Timestamp range         : {first_timestamp} → {last_timestamp}")

# ------------------------------------------------------------
# Create summary DataFrame
# ------------------------------------------------------------

timestamp_summary = pd.DataFrame(processing_summary)

print("\n" + "=" * 70)
print("PHASE 2.2 COMPLETED")
print("=" * 70)

display(
    timestamp_summary[
        [
            "File",
            "Input Rows",
            "Valid Rows",
            "Malformed Header Rows Removed",
            "Invalid Timestamp Rows Removed",
            "First Valid Timestamp",
            "Last Valid Timestamp"
        ]
    ]
)

print("\nProcessed files saved in:")
print(OUTPUT_DIR)

print("\nImportant:")
print("- Suspicious 1970 timestamps were NOT removed.")
print("- Duplicate rows were NOT removed.")
print("- Missing/Inf values were NOT handled.")
print("- Feature selection was NOT performed.")

PHASE 2.2 — TIMESTAMP CONVERSION & MALFORMED ROW HANDLING

[1/10] Processing:
Friday-02-03-2018_TrafficForML_CICFlowMeter.csv
  Input rows              : 1,048,575
  Valid rows              : 1,048,575
  Malformed header rows   : 0
  Invalid timestamp rows  : 0
  Timestamp range         : 2018-03-02 01:00:00 → 2018-03-02 12:59:59

[2/10] Processing:
Friday-16-02-2018_TrafficForML_CICFlowMeter.csv
  Input rows              : 1,048,575
  Valid rows              : 1,048,574
  Malformed header rows   : 1
  Invalid timestamp rows  : 0
  Timestamp range         : 2018-02-16 01:00:32 → 2018-02-16 12:58:24

[3/10] Processing:
Friday-23-02-2018_TrafficForML_CICFlowMeter.csv
  Input rows              : 1,048,575
  Valid rows              : 1,048,575
  Malformed header rows   : 0
  Invalid timestamp rows  : 0
  Timestamp range         : 2018-02-23 01:00:00 → 2018-02-23 12:59:59

[4/10] Processing:
Thuesday-20-02-2018_TrafficForML_CICFlowMeter.csv
  Input rows              : 7,948,748
  Valid rows

,File,Input Rows,Valid Rows,Malformed Header Rows Removed,Invalid Timestamp Rows Removed,First Valid Timestamp,Last Valid Timestamp
0,Friday-02-03-2018_TrafficForML_CICFlowMeter.csv,1048575,1048575,0,0,2018-03-02 01:00:00,2018-03-02 12:59:59
1,Friday-16-02-2018_TrafficForML_CICFlowMeter.csv,1048575,1048574,1,0,2018-02-16 01:00:32,2018-02-16 12:58:24
2,Friday-23-02-2018_TrafficForML_CICFlowMeter.csv,1048575,1048575,0,0,2018-02-23 01:00:00,2018-02-23 12:59:59
3,Thuesday-20-02-2018_TrafficForML_CICFlowMeter.csv,7948748,7948748,0,0,2018-02-20 01:00:00,2018-02-20 12:59:59
4,Thursday-01-03-2018_TrafficForML_CICFlowMeter.csv,331125,331100,25,0,2018-03-01 01:00:00,2018-03-01 12:59:59
5,Thursday-15-02-2018_TrafficForML_CICFlowMeter.csv,1048575,1048575,0,0,2018-02-15 01:00:00,2018-02-15 12:59:59
6,Thursday-22-02-2018_TrafficForML_CICFlowMeter.csv,1048575,1048575,0,0,1970-01-10 03:04:26,2018-02-22 12:59:59
7,Wednesday-14-02-2018_TrafficForML_CICFlowMeter...,1048575,1048575,0,0,1970-01-05 03:01:17,2018-02-14 12:59:59
8,Wednesday-21-02-2018_TrafficForML_CICFlowMeter...,1048575,1048575,0,0,2018-02-21 01:55:46,2018-02-21 10:43:21
9,Wednesday-28-02-2018_TrafficForML_CICFlowMeter...,613104,613071,33,0,2018-02-28 01:00:00,2018-02-28 12:59:59



Processed files saved in:
/content/phase2_processed

Important:
- Suspicious 1970 timestamps were NOT removed.
- Duplicate rows were NOT removed.
- Missing/Inf values were NOT handled.
- Feature selection was NOT performed.


## 2.3 — Suspicious Timestamp Investigation

Phase 1 identified timestamps around the year 1970 in two files:

- Thursday-22-02-2018
- Wednesday-14-02-2018

These timestamps are technically parseable but inconsistent with the
expected 2018 capture period.

This step investigates the affected records by examining their count,
timestamp range, labels, and selected traffic characteristics.

No records are removed or modified during this investigation.

In [11]:
# ============================================================
# PHASE 2 — DATA PREPROCESSING
# 2.3 — Suspicious Timestamp Investigation
# ============================================================

import os
import pandas as pd
import numpy as np

# ------------------------------------------------------------
# Files identified during Phase 1
# ------------------------------------------------------------

affected_files = [
    "Thursday-22-02-2018_TrafficForML_CICFlowMeter.csv",
    "Wednesday-14-02-2018_TrafficForML_CICFlowMeter.csv"
]

# Use the already processed files from Phase 2.2
affected_paths = {
    file_name: os.path.join(OUTPUT_DIR, file_name)
    for file_name in affected_files
}

# Expected capture year
EXPECTED_YEAR = 2018

print("=" * 70)
print("PHASE 2.3 — SUSPICIOUS TIMESTAMP INVESTIGATION")
print("=" * 70)

investigation_summary = []

for file_name, file_path in affected_paths.items():

    print("\n" + "-" * 70)
    print(f"FILE: {file_name}")
    print("-" * 70)

    suspicious_count = 0
    suspicious_labels = {}
    suspicious_timestamps = []

    # Selected traffic features for basic inspection
    selected_features = [
        "Flow Duration",
        "Tot Fwd Pkts",
        "Tot Bwd Pkts",
        "Flow Byts/s",
        "Flow Pkts/s"
    ]

    feature_records = []

    # --------------------------------------------------------
    # Read in chunks
    # --------------------------------------------------------

    for chunk in pd.read_csv(
        file_path,
        chunksize=100_000,
        low_memory=False
    ):

        # Convert timestamp
        chunk["Timestamp"] = pd.to_datetime(
            chunk["Timestamp"],
            format="%Y-%m-%d %H:%M:%S",
            errors="coerce"
        )

        # Identify timestamps outside expected year
        suspicious_mask = (
            chunk["Timestamp"].notna()
            & (chunk["Timestamp"].dt.year != EXPECTED_YEAR)
        )

        suspicious_chunk = chunk.loc[suspicious_mask].copy()

        if suspicious_chunk.empty:
            continue

        suspicious_count += len(suspicious_chunk)

        # ----------------------------------------------------
        # Collect timestamp information
        # ----------------------------------------------------

        suspicious_timestamps.extend(
            suspicious_chunk["Timestamp"].tolist()
        )

        # ----------------------------------------------------
        # Collect label distribution
        # ----------------------------------------------------

        label_counts = (
            suspicious_chunk["Label"]
            .astype(str)
            .value_counts()
        )

        for label, count in label_counts.items():
            suspicious_labels[label] = (
                suspicious_labels.get(label, 0) + int(count)
            )

        # ----------------------------------------------------
        # Collect selected feature values
        # ----------------------------------------------------

        available_features = [
            feature
            for feature in selected_features
            if feature in suspicious_chunk.columns
        ]

        if available_features:
            feature_records.append(
                suspicious_chunk[available_features]
            )

    # --------------------------------------------------------
    # Display investigation results
    # --------------------------------------------------------

    if suspicious_count == 0:

        print("No suspicious timestamps found.")

        investigation_summary.append({
            "File": file_name,
            "Suspicious Rows": 0,
            "Minimum Suspicious Timestamp": None,
            "Maximum Suspicious Timestamp": None
        })

        continue

    timestamp_series = pd.Series(suspicious_timestamps)

    print(f"Suspicious rows        : {suspicious_count:,}")
    print(
        f"Timestamp range        : "
        f"{timestamp_series.min()} → {timestamp_series.max()}"
    )

    # --------------------------------------------------------
    # Show exact timestamp frequencies
    # --------------------------------------------------------

    print("\nSuspicious timestamp frequencies:")

    timestamp_counts = (
        timestamp_series
        .value_counts()
        .sort_index()
    )

    display(
        timestamp_counts
        .rename("Count")
        .to_frame()
        .head(20)
    )

    # --------------------------------------------------------
    # Show label distribution
    # --------------------------------------------------------

    print("Labels associated with suspicious timestamps:")

    label_df = (
        pd.Series(suspicious_labels)
        .sort_values(ascending=False)
        .rename("Count")
        .to_frame()
    )

    label_df["Percentage"] = (
        label_df["Count"] / suspicious_count * 100
    )

    display(label_df)

    # --------------------------------------------------------
    # Show selected traffic-feature statistics
    # --------------------------------------------------------

    if feature_records:

        feature_df = pd.concat(
            feature_records,
            ignore_index=True
        )

        print("Selected traffic-feature statistics:")

        display(
            feature_df.describe()
        )

    # --------------------------------------------------------
    # Save summary
    # --------------------------------------------------------

    investigation_summary.append({
        "File": file_name,
        "Suspicious Rows": suspicious_count,
        "Minimum Suspicious Timestamp": str(timestamp_series.min()),
        "Maximum Suspicious Timestamp": str(timestamp_series.max())
    })


# ------------------------------------------------------------
# Final summary
# ------------------------------------------------------------

print("\n" + "=" * 70)
print("SUSPICIOUS TIMESTAMP INVESTIGATION SUMMARY")
print("=" * 70)

summary_df = pd.DataFrame(investigation_summary)

display(summary_df)

print("\nNo records were deleted or modified in this step.")
print("The results will determine the timestamp-cleaning decision.")

PHASE 2.3 — SUSPICIOUS TIMESTAMP INVESTIGATION

----------------------------------------------------------------------
FILE: Thursday-22-02-2018_TrafficForML_CICFlowMeter.csv
----------------------------------------------------------------------
Suspicious rows        : 9
Timestamp range        : 1970-01-10 03:04:26 → 1970-01-12 09:30:26

Suspicious timestamp frequencies:


,Count
1970-01-10 03:04:26,1
1970-01-11 03:51:32,1
1970-01-11 05:12:30,1
1970-01-11 12:05:36,1
1970-01-12 01:09:53,1
1970-01-12 06:40:49,1
1970-01-12 09:18:52,1
1970-01-12 09:30:03,1
1970-01-12 09:30:26,1


Labels associated with suspicious timestamps:


,Count,Percentage
Benign,9,100.0


Selected traffic-feature statistics:


,Flow Duration,Tot Fwd Pkts,Tot Bwd Pkts,Flow Byts/s,Flow Pkts/s
count,9.000000e+00,9.000000,9.0,9.0,9.000000
mean,-4.088600e+11,47.111111,0.0,0.0,-0.001136
std,3.116039e+11,49.670022,0.0,0.0,0.002918
min,-8.282200e+11,2.000000,0.0,0.0,-0.008895
25%,-6.420520e+11,9.000000,0.0,0.0,-0.000254
50%,-5.533120e+11,38.000000,0.0,0.0,-0.000083
75%,-1.101160e+11,48.000000,0.0,0.0,-0.000054
max,-4.834000e+09,163.000000,0.0,0.0,-0.000002


... (5 outputs left out)

,Flow Duration,Tot Fwd Pkts,Tot Bwd Pkts,Flow Byts/s,Flow Pkts/s
count,5.000000e+00,5.000000,5.0,5.0,5.000000
mean,-4.831874e+11,174.600000,0.0,0.0,-0.000449
std,3.526200e+11,217.196455,0.0,0.0,0.000500
min,-9.190110e+11,2.000000,0.0,0.0,-0.001030
25%,-6.814020e+11,3.000000,0.0,0.0,-0.000953
50%,-5.298010e+11,81.000000,0.0,0.0,-0.000168
75%,-2.738500e+11,282.000000,0.0,0.0,-0.000088
max,-1.187300e+10,505.000000,0.0,0.0,-0.000004



SUSPICIOUS TIMESTAMP INVESTIGATION SUMMARY


,File,Suspicious Rows,Minimum Suspicious Timestamp,Maximum Suspicious Timestamp
0,Thursday-22-02-2018_TrafficForML_CICFlowMeter.csv,9,1970-01-10 03:04:26,1970-01-12 09:30:26
1,Wednesday-14-02-2018_TrafficForML_CICFlowMeter...,5,1970-01-05 03:01:17,1970-01-12 09:44:12



No records were deleted or modified in this step.
The results will determine the timestamp-cleaning decision.


## 2.4 — Timestamp Cleaning & Chronological Ordering

The suspicious timestamp investigation identified 14 records with timestamps
from 1970 instead of the expected 2018 capture period. These records also
contained abnormal traffic values and are therefore removed.


After removing these records, the remaining traffic records are ordered
chronologically using the `Timestamp` column.


Chronological ordering is essential for the network attack forecasting task
because future-state prediction must respect the actual temporal sequence of
network traffic.


The original row order of the raw CSV files is not assumed to represent
chronological order.

In [12]:
# ============================================================
# PHASE 2 — DATA PREPROCESSING
# 2.4 — TIMESTAMP CLEANING & CHRONOLOGICAL ORDERING
# ============================================================

# Install DuckDB if it is not already available.
!pip -q install duckdb

import os
import duckdb
import pandas as pd

# ------------------------------------------------------------
# Configuration
# ------------------------------------------------------------

INPUT_DIR = "/content/phase2_processed"
OUTPUT_DIR = "/content/phase2_chronological"

os.makedirs(OUTPUT_DIR, exist_ok=True)

EXPECTED_YEAR = 2018

print("=" * 70)
print("PHASE 2.4 — TIMESTAMP CLEANING & CHRONOLOGICAL ORDERING")
print("=" * 70)

# ------------------------------------------------------------
# Files to process
# ------------------------------------------------------------

files_to_process = list(dataset_paths.keys())

ordering_results = []

# ------------------------------------------------------------
# Process each file
# ------------------------------------------------------------

for file_number, file_name in enumerate(files_to_process, start=1):

    input_path = os.path.join(INPUT_DIR, file_name)
    output_path = os.path.join(OUTPUT_DIR, file_name)

    print(f"\n[{file_number}/10] Processing:")
    print(file_name)

    # --------------------------------------------------------
    # Remove previous output if the cell is re-run
    # --------------------------------------------------------

    if os.path.exists(output_path):
        os.remove(output_path)

    # --------------------------------------------------------
    # Create a fresh DuckDB connection
    #
    # DuckDB performs the ORDER BY operation using its
    # disk-based execution system, avoiding the need to
    # load the entire file into pandas RAM.
    # --------------------------------------------------------

    con = duckdb.connect()

    try:

        # ----------------------------------------------------
        # Read the CSV, remove suspicious years, and sort.
        #
        # The Timestamp in Phase 2.2 was saved as:
        # YYYY-MM-DD HH:MM:SS
        # ----------------------------------------------------

        query = f"""
        COPY (
            SELECT *
            FROM read_csv_auto(
                '{input_path}',
                header = true,
                ignore_errors = false
            )
            WHERE
                TRY_CAST("Timestamp" AS TIMESTAMP) IS NOT NULL
                AND EXTRACT(
                    YEAR FROM TRY_CAST("Timestamp" AS TIMESTAMP)
                ) = {EXPECTED_YEAR}
            ORDER BY
                TRY_CAST("Timestamp" AS TIMESTAMP)
        )
        TO '{output_path}'
        WITH (
            HEADER,
            DELIMITER ','
        );
        """

        con.execute(query)

        # ----------------------------------------------------
        # Obtain output row count
        # ----------------------------------------------------

        count_query = f"""
        SELECT COUNT(*)
        FROM read_csv_auto(
            '{output_path}',
            header = true
        );
        """

        output_rows = con.execute(count_query).fetchone()[0]

    finally:
        con.close()

    # --------------------------------------------------------
    # Determine rows removed
    #
    # Input row count comes from Phase 2.2 metadata processing.
    # --------------------------------------------------------

    matching_metadata = [
        record
        for record in metadata_records
        if record["file_name"] == file_name
    ]

    input_rows = (
        matching_metadata[0]["number_of_rows"]
        if matching_metadata
        else None
    )

    rows_removed = (
        input_rows - output_rows
        if input_rows is not None
        else None
    )

    # --------------------------------------------------------
    # Verify chronological ordering using chunks
    # --------------------------------------------------------

    chronological = True
    verification_rows = 0
    previous_timestamp = None

    for chunk in pd.read_csv(
        output_path,
        usecols=["Timestamp"],
        chunksize=100_000
    ):

        timestamps = pd.to_datetime(
            chunk["Timestamp"],
            format="%Y-%m-%d %H:%M:%S",
            errors="coerce"
        )

        # Check within current chunk
        if not timestamps.is_monotonic_increasing:
            chronological = False
            break

        # Check boundary between previous and current chunk
        if (
            previous_timestamp is not None
            and not timestamps.empty
            and timestamps.iloc[0] >= previous_timestamp
        ):
            pass
        elif (
            previous_timestamp is not None
            and not timestamps.empty
            and timestamps.iloc[0] < previous_timestamp
        ):
            chronological = False
            break

        if not timestamps.empty:
            previous_timestamp = timestamps.iloc[-1]

        verification_rows += len(chunk)

    # --------------------------------------------------------
    # Obtain final timestamp range
    # --------------------------------------------------------

    range_df = pd.read_csv(
        output_path,
        usecols=["Timestamp"],
        nrows=1
    )

    first_timestamp = range_df["Timestamp"].iloc[0]

    # Efficiently get last timestamp by reading only Timestamp
    # column in chunks.
    last_timestamp = None

    for chunk in pd.read_csv(
        output_path,
        usecols=["Timestamp"],
        chunksize=100_000
    ):
        if not chunk.empty:
            last_timestamp = chunk["Timestamp"].iloc[-1]

    # --------------------------------------------------------
    # Store results
    # --------------------------------------------------------

    ordering_results.append({
        "File": file_name,
        "Input Rows": input_rows,
        "Output Rows": output_rows,
        "Rows Removed": rows_removed,
        "First Timestamp": first_timestamp,
        "Last Timestamp": last_timestamp,
        "Chronological": chronological
    })

    print(f"  Input rows       : {input_rows:,}")
    print(f"  Output rows      : {output_rows:,}")
    print(f"  Rows removed     : {rows_removed:,}")
    print(f"  First timestamp  : {first_timestamp}")
    print(f"  Last timestamp   : {last_timestamp}")
    print(f"  Chronological    : {chronological}")

# ------------------------------------------------------------
# Final summary
# ------------------------------------------------------------

ordering_summary = pd.DataFrame(ordering_results)

print("\n" + "=" * 70)
print("PHASE 2.4 COMPLETED")
print("=" * 70)

display(ordering_summary)

print("\nChronologically ordered files saved in:")
print(OUTPUT_DIR)

print("\nValidation:")
print(
    "All files chronological:",
    bool(ordering_summary["Chronological"].all())
)

print(
    "Total rows removed:",
    int(ordering_summary["Rows Removed"].sum())
)

PHASE 2.4 — TIMESTAMP CLEANING & CHRONOLOGICAL ORDERING

[1/10] Processing:
Friday-02-03-2018_TrafficForML_CICFlowMeter.csv


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

  Input rows       : 1,048,575
  Output rows      : 1,048,575
  Rows removed     : 0
  First timestamp  : 2018-03-02 01:00:00
  Last timestamp   : 2018-03-02 12:59:59
  Chronological    : True

[2/10] Processing:
Friday-16-02-2018_TrafficForML_CICFlowMeter.csv


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

  Input rows       : 1,048,575
  Output rows      : 1,048,574
  Rows removed     : 1
  First timestamp  : 2018-02-16 01:00:32
  Last timestamp   : 2018-02-16 12:58:24
  Chronological    : True

[3/10] Processing:
Friday-23-02-2018_TrafficForML_CICFlowMeter.csv


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

... (14 outputs left out)

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

  Input rows       : 613,104
  Output rows      : 613,071
  Rows removed     : 33
  First timestamp  : 2018-02-28 01:00:00
  Last timestamp   : 2018-02-28 12:59:59
  Chronological    : True

PHASE 2.4 COMPLETED


,File,Input Rows,Output Rows,Rows Removed,First Timestamp,Last Timestamp,Chronological
0,Friday-02-03-2018_TrafficForML_CICFlowMeter.csv,1048575,1048575,0,2018-03-02 01:00:00,2018-03-02 12:59:59,True
1,Friday-16-02-2018_TrafficForML_CICFlowMeter.csv,1048575,1048574,1,2018-02-16 01:00:32,2018-02-16 12:58:24,True
2,Friday-23-02-2018_TrafficForML_CICFlowMeter.csv,1048575,1048575,0,2018-02-23 01:00:00,2018-02-23 12:59:59,True
3,Thuesday-20-02-2018_TrafficForML_CICFlowMeter.csv,7948748,7948748,0,2018-02-20 01:00:00,2018-02-20 12:59:59,True
4,Thursday-01-03-2018_TrafficForML_CICFlowMeter.csv,331125,331100,25,2018-03-01 01:00:00,2018-03-01 12:59:59,True
5,Thursday-15-02-2018_TrafficForML_CICFlowMeter.csv,1048575,1048575,0,2018-02-15 01:00:00,2018-02-15 12:59:59,True
6,Thursday-22-02-2018_TrafficForML_CICFlowMeter.csv,1048575,1048566,9,2018-02-22 01:00:00,2018-02-22 12:59:59,True
7,Wednesday-14-02-2018_TrafficForML_CICFlowMeter...,1048575,1048570,5,2018-02-14 01:00:00,2018-02-14 12:59:59,True
8,Wednesday-21-02-2018_TrafficForML_CICFlowMeter...,1048575,1048575,0,2018-02-21 01:55:46,2018-02-21 10:43:21,True
9,Wednesday-28-02-2018_TrafficForML_CICFlowMeter...,613104,613071,33,2018-02-28 01:00:00,2018-02-28 12:59:59,True



Chronologically ordered files saved in:
/content/phase2_chronological

Validation:
All files chronological: True
Total rows removed: 73


## 2.5 — Missing & Infinite Value Handling

The CIC-IDS2018 dataset contains missing and infinite values, particularly
in flow-rate features such as `Flow Byts/s` and `Flow Pkts/s`.


Infinite values are not valid numerical observations for machine-learning
models, so they are converted to missing values (`NaN`) during preprocessing.


The affected features and their frequencies are recorded before deciding
the final imputation or removal strategy.


No arbitrary zero or median imputation is applied without first examining
the resulting missing-value distribution.

In [14]:
# ============================================================
# PHASE 2 — DATA PREPROCESSING
# 2.5 — MISSING & INFINITE VALUE HANDLING
# ============================================================

import os
import numpy as np
import pandas as pd

# ------------------------------------------------------------
# Configuration
# ------------------------------------------------------------

INPUT_DIR = "/content/phase2_chronological"
OUTPUT_DIR = "/content/phase2_quality_cleaned"

os.makedirs(OUTPUT_DIR, exist_ok=True)

CHUNK_SIZE = 100_000

print("=" * 70)
print("PHASE 2.5 — MISSING & INFINITE VALUE HANDLING")
print("=" * 70)

quality_summary = []
feature_quality_summary = []

# ------------------------------------------------------------
# Process all files
# ------------------------------------------------------------

for file_number, file_name in enumerate(files_to_process, start=1):

    input_path = os.path.join(INPUT_DIR, file_name)
    output_path = os.path.join(OUTPUT_DIR, file_name)

    print(f"\n[{file_number}/10] Processing:")
    print(file_name)

    # Remove previous output if notebook cell is re-run
    if os.path.exists(output_path):
        os.remove(output_path)

    total_rows = 0
    missing_cells_before = 0
    infinite_cells = 0
    missing_cells_after_inf_conversion = 0

    missing_by_column = {}
    infinite_by_column = {}

    first_chunk = True

    # --------------------------------------------------------
    # Read file in chunks
    # --------------------------------------------------------

    for chunk in pd.read_csv(
        input_path,
        chunksize=CHUNK_SIZE,
        low_memory=False
    ):

        total_rows += len(chunk)

        # ----------------------------------------------------
        # Identify numerical columns
        # ----------------------------------------------------

        numeric_columns = chunk.select_dtypes(
            include=[np.number]
        ).columns

        # ----------------------------------------------------
        # Count missing values
        # ----------------------------------------------------

        missing_counts = chunk.isna().sum()

        for column, count in missing_counts.items():

            count = int(count)

            if count > 0:
                missing_by_column[column] = (
                    missing_by_column.get(column, 0) + count
                )

                missing_cells_before += count

        # ----------------------------------------------------
        # Detect infinite values
        # ----------------------------------------------------

        if len(numeric_columns) > 0:

            numeric_data = chunk[numeric_columns]

            positive_inf = np.isposinf(numeric_data)
            negative_inf = np.isneginf(numeric_data)

            inf_counts = (
                positive_inf.sum()
                + negative_inf.sum()
            )

            for column, count in inf_counts.items():

                count = int(count)

                if count > 0:
                    infinite_by_column[column] = (
                        infinite_by_column.get(column, 0) + count
                    )

                    infinite_cells += count

            # ------------------------------------------------
            # Convert +/- infinity to NaN
            # ------------------------------------------------

            chunk[numeric_columns] = (
                chunk[numeric_columns]
                .replace([np.inf, -np.inf], np.nan)
            )

        # ----------------------------------------------------
        # Count missing values after infinity conversion
        # ----------------------------------------------------

        missing_after = chunk.isna().sum()

        missing_cells_after_inf_conversion += int(
            missing_after.sum()
        )

        # ----------------------------------------------------
        # Save processed chunk
        # ----------------------------------------------------

        chunk.to_csv(
            output_path,
            mode="w" if first_chunk else "a",
            header=first_chunk,
            index=False
        )

        first_chunk = False

    # --------------------------------------------------------
    # Feature-level summary
    # --------------------------------------------------------

    affected_features = sorted(
        set(missing_by_column.keys())
        | set(infinite_by_column.keys())
    )

    for feature in affected_features:

        feature_quality_summary.append({
            "File": file_name,
            "Feature": feature,
            "Missing Before": missing_by_column.get(
                feature, 0
            ),
            "Infinite": infinite_by_column.get(
                feature, 0
            ),
            "Missing After Inf Conversion": (
                missing_by_column.get(feature, 0)
                + infinite_by_column.get(feature, 0)
            )
        })

    # --------------------------------------------------------
    # File-level summary
    # --------------------------------------------------------

    quality_summary.append({
        "File": file_name,
        "Rows": total_rows,
        "Missing Cells Before": missing_cells_before,
        "Infinite Cells": infinite_cells,
        "Missing Cells After Inf Conversion":
            missing_cells_after_inf_conversion,
        "Affected Features": len(affected_features)
    })

    print(f"  Rows                         : {total_rows:,}")
    print(f"  Missing cells before        : {missing_cells_before:,}")
    print(f"  Infinite cells              : {infinite_cells:,}")
    print(
        f"  Missing after Inf conversion: "
        f"{missing_cells_after_inf_conversion:,}"
    )
    print(f"  Affected features           : {len(affected_features)}")

# ------------------------------------------------------------
# Create summary tables
# ------------------------------------------------------------

quality_df = pd.DataFrame(quality_summary)

feature_quality_df = pd.DataFrame(
    feature_quality_summary
)

# ------------------------------------------------------------
# Display file-level summary
# ------------------------------------------------------------

print("\n" + "=" * 70)
print("FILE-LEVEL QUALITY SUMMARY")
print("=" * 70)

display(quality_df)

# ------------------------------------------------------------
# Display feature-level summary
# ------------------------------------------------------------

print("\n" + "=" * 70)
print("FEATURE-LEVEL MISSING / INFINITE SUMMARY")
print("=" * 70)

if not feature_quality_df.empty:

    display(
        feature_quality_df.sort_values(
            ["File", "Missing After Inf Conversion"],
            ascending=[True, False]
        )
    )

else:

    print("No missing or infinite values detected.")

# ------------------------------------------------------------
# Overall totals
# ------------------------------------------------------------

print("\n" + "=" * 70)
print("OVERALL RESULTS")
print("=" * 70)

print(
    "Total missing cells before conversion:",
    f"{quality_df['Missing Cells Before'].sum():,}"
)

print(
    "Total infinite cells:",
    f"{quality_df['Infinite Cells'].sum():,}"
)

print(
    "Total missing cells after infinity conversion:",
    f"{quality_df['Missing Cells After Inf Conversion'].sum():,}"
)

print("\nProcessed files saved in:")
print(OUTPUT_DIR)

print("\nImportant:")
print("- Infinite values have been converted to NaN.")
print("- Missing values have NOT been imputed.")
print("- No rows were removed because of missing values.")
print("- No SMOTE or resampling has been applied.")

PHASE 2.5 — MISSING & INFINITE VALUE HANDLING

[1/10] Processing:
Friday-02-03-2018_TrafficForML_CICFlowMeter.csv
  Rows                         : 1,048,575
  Missing cells before        : 2,558
  Infinite cells              : 5,542
  Missing after Inf conversion: 8,100
  Affected features           : 2

[2/10] Processing:
Friday-16-02-2018_TrafficForML_CICFlowMeter.csv
  Rows                         : 1,048,574
  Missing cells before        : 0
  Infinite cells              : 0
  Missing after Inf conversion: 0
  Affected features           : 0

[3/10] Processing:
Friday-23-02-2018_TrafficForML_CICFlowMeter.csv
  Rows                         : 1,048,575
  Missing cells before        : 3,754
  Infinite cells              : 7,662
  Missing after Inf conversion: 11,416
  Affected features           : 2

[4/10] Processing:
Thuesday-20-02-2018_TrafficForML_CICFlowMeter.csv
  Rows                         : 7,948,748
  Missing cells before        : 36,767
  Infinite cells              : 82,1

,File,Rows,Missing Cells Before,Infinite Cells,Missing Cells After Inf Conversion,Affected Features
0,Friday-02-03-2018_TrafficForML_CICFlowMeter.csv,1048575,2558,5542,8100,2
1,Friday-16-02-2018_TrafficForML_CICFlowMeter.csv,1048574,0,0,0,0
2,Friday-23-02-2018_TrafficForML_CICFlowMeter.csv,1048575,3754,7662,11416,2
3,Thuesday-20-02-2018_TrafficForML_CICFlowMeter.csv,7948748,36767,82139,118906,2
4,Thursday-01-03-2018_TrafficForML_CICFlowMeter.csv,331100,1834,4004,5838,2
5,Thursday-15-02-2018_TrafficForML_CICFlowMeter.csv,1048575,4921,11133,16054,2
6,Thursday-22-02-2018_TrafficForML_CICFlowMeter.csv,1048566,3569,7651,11220,2
7,Wednesday-14-02-2018_TrafficForML_CICFlowMeter...,1048570,2277,5371,7648,2
8,Wednesday-21-02-2018_TrafficForML_CICFlowMeter...,1048575,0,0,0,0
9,Wednesday-28-02-2018_TrafficForML_CICFlowMeter...,613071,4041,8297,12338,2



FEATURE-LEVEL MISSING / INFINITE SUMMARY


,File,Feature,Missing Before,Infinite,Missing After Inf Conversion
0,Friday-02-03-2018_TrafficForML_CICFlowMeter.csv,Flow Byts/s,2558,1492,4050
1,Friday-02-03-2018_TrafficForML_CICFlowMeter.csv,Flow Pkts/s,0,4050,4050
2,Friday-23-02-2018_TrafficForML_CICFlowMeter.csv,Flow Byts/s,3754,1954,5708
3,Friday-23-02-2018_TrafficForML_CICFlowMeter.csv,Flow Pkts/s,0,5708,5708
4,Thuesday-20-02-2018_TrafficForML_CICFlowMeter.csv,Flow Byts/s,36767,22686,59453
5,Thuesday-20-02-2018_TrafficForML_CICFlowMeter.csv,Flow Pkts/s,0,59453,59453
6,Thursday-01-03-2018_TrafficForML_CICFlowMeter.csv,Flow Byts/s,1834,1085,2919
7,Thursday-01-03-2018_TrafficForML_CICFlowMeter.csv,Flow Pkts/s,0,2919,2919
8,Thursday-15-02-2018_TrafficForML_CICFlowMeter.csv,Flow Byts/s,4921,3106,8027
9,Thursday-15-02-2018_TrafficForML_CICFlowMeter.csv,Flow Pkts/s,0,8027,8027



OVERALL RESULTS
Total missing cells before conversion: 59,721
Total infinite cells: 131,799
Total missing cells after infinity conversion: 191,520

Processed files saved in:
/content/phase2_quality_cleaned

Important:
- Infinite values have been converted to NaN.
- Missing values have NOT been imputed.
- No rows were removed because of missing values.
- No SMOTE or resampling has been applied.


## 2.6 — Duplicate Record Investigation

Exact duplicate records were identified during Phase 1, including relatively
high duplicate proportions in some capture files.

Before removing duplicates, their characteristics are investigated to
determine whether they are likely to represent redundant observations.

The investigation examines:

- Exact duplicate count
- Duplicate percentage
- Labels associated with duplicate records
- Whether duplicate records share the same timestamp
- Duplicate distribution across benign and attack traffic

No duplicate records are removed during this step.

In [15]:
# ============================================================
# PHASE 2 — DATA PREPROCESSING
# 2.6 — DUPLICATE RECORD INVESTIGATION
# ============================================================

import os
import pandas as pd

# ------------------------------------------------------------
# Configuration
# ------------------------------------------------------------

INPUT_DIR = "/content/phase2_quality_cleaned"

CHUNK_SIZE = 100_000

print("=" * 70)
print("PHASE 2.6 — DUPLICATE RECORD INVESTIGATION")
print("=" * 70)

duplicate_summary = []
duplicate_label_summary = []

# ------------------------------------------------------------
# Investigate each file
# ------------------------------------------------------------

for file_number, file_name in enumerate(files_to_process, start=1):

    file_path = os.path.join(INPUT_DIR, file_name)

    print(f"\n[{file_number}/10] Investigating:")
    print(file_name)

    # --------------------------------------------------------
    # For exact duplicate detection across a large CSV,
    # maintain a hash set of complete rows.
    #
    # This avoids storing complete duplicate DataFrames,
    # although the hash set itself can become large.
    # --------------------------------------------------------

    seen_hashes = set()

    total_rows = 0
    duplicate_rows = 0

    duplicate_label_counts = {}
    duplicate_timestamp_counts = {}

    # --------------------------------------------------------
    # Read in chunks
    # --------------------------------------------------------

    for chunk in pd.read_csv(
        file_path,
        chunksize=CHUNK_SIZE,
        low_memory=False
    ):

        total_rows += len(chunk)

        # ----------------------------------------------------
        # Generate a hash for every complete row
        # ----------------------------------------------------

        row_hashes = pd.util.hash_pandas_object(
            chunk,
            index=False
        )

        # ----------------------------------------------------
        # Detect duplicates both within the current chunk
        # and against previous chunks.
        # ----------------------------------------------------

        duplicate_mask = []

        for row_hash in row_hashes:

            row_hash = int(row_hash)

            if row_hash in seen_hashes:
                duplicate_mask.append(True)
                duplicate_rows += 1
            else:
                duplicate_mask.append(False)
                seen_hashes.add(row_hash)

        duplicate_mask = pd.Series(
            duplicate_mask,
            index=chunk.index
        )

        duplicate_chunk = chunk.loc[
            duplicate_mask
        ]

        # ----------------------------------------------------
        # Label distribution among duplicate records
        # ----------------------------------------------------

        if not duplicate_chunk.empty:

            label_counts = (
                duplicate_chunk["Label"]
                .astype(str)
                .value_counts()
            )

            for label, count in label_counts.items():

                duplicate_label_counts[label] = (
                    duplicate_label_counts.get(label, 0)
                    + int(count)
                )

            # ------------------------------------------------
            # Timestamp distribution
            # ------------------------------------------------

            timestamp_counts = (
                duplicate_chunk["Timestamp"]
                .astype(str)
                .value_counts()
            )

            for timestamp, count in timestamp_counts.items():

                duplicate_timestamp_counts[timestamp] = (
                    duplicate_timestamp_counts.get(timestamp, 0)
                    + int(count)
                )

    # --------------------------------------------------------
    # Calculate duplicate percentage
    # --------------------------------------------------------

    duplicate_percentage = (
        duplicate_rows / total_rows * 100
        if total_rows > 0
        else 0
    )

    # --------------------------------------------------------
    # Save file-level summary
    # --------------------------------------------------------

    duplicate_summary.append({
        "File": file_name,
        "Total Rows": total_rows,
        "Exact Duplicate Rows": duplicate_rows,
        "Duplicate Percentage": duplicate_percentage
    })

    # --------------------------------------------------------
    # Save duplicate label distribution
    # --------------------------------------------------------

    for label, count in duplicate_label_counts.items():

        duplicate_label_summary.append({
            "File": file_name,
            "Label": label,
            "Duplicate Rows": count,
            "Percentage of Duplicates":
                count / duplicate_rows * 100
                if duplicate_rows > 0 else 0
        })

    # --------------------------------------------------------
    # Display file result
    # --------------------------------------------------------

    print(f"  Total rows          : {total_rows:,}")
    print(f"  Exact duplicates    : {duplicate_rows:,}")
    print(f"  Duplicate percentage: {duplicate_percentage:.4f}%")

# ------------------------------------------------------------
# Create summary DataFrames
# ------------------------------------------------------------

duplicate_df = pd.DataFrame(
    duplicate_summary
)

duplicate_label_df = pd.DataFrame(
    duplicate_label_summary
)

# ------------------------------------------------------------
# Display file-level results
# ------------------------------------------------------------

print("\n" + "=" * 70)
print("DUPLICATE SUMMARY")
print("=" * 70)

display(duplicate_df)

# ------------------------------------------------------------
# Display label distribution
# ------------------------------------------------------------

print("\n" + "=" * 70)
print("DUPLICATE LABEL DISTRIBUTION")
print("=" * 70)

if not duplicate_label_df.empty:

    display(
        duplicate_label_df.sort_values(
            ["File", "Duplicate Rows"],
            ascending=[True, False]
        )
    )

else:

    print("No duplicate records detected.")

# ------------------------------------------------------------
# Overall duplicate statistics
# ------------------------------------------------------------

total_dataset_rows = duplicate_df["Total Rows"].sum()
total_duplicate_rows = duplicate_df["Exact Duplicate Rows"].sum()

overall_duplicate_percentage = (
    total_duplicate_rows / total_dataset_rows * 100
)

print("\n" + "=" * 70)
print("OVERALL DUPLICATE RESULTS")
print("=" * 70)

print(
    "Total rows investigated:",
    f"{total_dataset_rows:,}"
)

print(
    "Total exact duplicate rows:",
    f"{total_duplicate_rows:,}"
)

print(
    "Overall duplicate percentage:",
    f"{overall_duplicate_percentage:.4f}%"
)

print("\nNo duplicate records were removed in this step.")

PHASE 2.6 — DUPLICATE RECORD INVESTIGATION

[1/10] Investigating:
Friday-02-03-2018_TrafficForML_CICFlowMeter.csv
  Total rows          : 1,048,575
  Exact duplicates    : 5,459
  Duplicate percentage: 0.5206%

[2/10] Investigating:
Friday-16-02-2018_TrafficForML_CICFlowMeter.csv
  Total rows          : 1,048,574
  Exact duplicates    : 147,586
  Duplicate percentage: 14.0749%

[3/10] Investigating:
Friday-23-02-2018_TrafficForML_CICFlowMeter.csv
  Total rows          : 1,048,575
  Exact duplicates    : 2,614
  Duplicate percentage: 0.2493%

[4/10] Investigating:
Thuesday-20-02-2018_TrafficForML_CICFlowMeter.csv
  Total rows          : 7,948,748
  Exact duplicates    : 2
  Duplicate percentage: 0.0000%

[5/10] Investigating:
Thursday-01-03-2018_TrafficForML_CICFlowMeter.csv
  Total rows          : 331,100
  Exact duplicates    : 73
  Duplicate percentage: 0.0220%

[6/10] Investigating:
Thursday-15-02-2018_TrafficForML_CICFlowMeter.csv
  Total rows          : 1,048,575
  Exact duplicate

,File,Total Rows,Exact Duplicate Rows,Duplicate Percentage
0,Friday-02-03-2018_TrafficForML_CICFlowMeter.csv,1048575,5459,0.520611
1,Friday-16-02-2018_TrafficForML_CICFlowMeter.csv,1048574,147586,14.074925
2,Friday-23-02-2018_TrafficForML_CICFlowMeter.csv,1048575,2614,0.249291
3,Thuesday-20-02-2018_TrafficForML_CICFlowMeter.csv,7948748,2,0.000025
4,Thursday-01-03-2018_TrafficForML_CICFlowMeter.csv,331100,73,0.022048
5,Thursday-15-02-2018_TrafficForML_CICFlowMeter.csv,1048575,2421,0.230885
6,Thursday-22-02-2018_TrafficForML_CICFlowMeter.csv,1048566,3278,0.312617
7,Wednesday-14-02-2018_TrafficForML_CICFlowMeter...,1048570,225628,21.517686
8,Wednesday-21-02-2018_TrafficForML_CICFlowMeter...,1048575,17557,1.674368
9,Wednesday-28-02-2018_TrafficForML_CICFlowMeter...,613071,6089,0.993197



DUPLICATE LABEL DISTRIBUTION


,File,Label,Duplicate Rows,Percentage of Duplicates
1,Friday-02-03-2018_TrafficForML_CICFlowMeter.csv,Bot,3881,71.093607
0,Friday-02-03-2018_TrafficForML_CICFlowMeter.csv,Benign,1578,28.906393
3,Friday-16-02-2018_TrafficForML_CICFlowMeter.csv,DoS attacks-SlowHTTPTest,120428,81.598526
2,Friday-16-02-2018_TrafficForML_CICFlowMeter.csv,DoS attacks-Hulk,27039,18.320843
4,Friday-16-02-2018_TrafficForML_CICFlowMeter.csv,Benign,119,0.080631
5,Friday-23-02-2018_TrafficForML_CICFlowMeter.csv,Benign,2614,100.000000
6,Thuesday-20-02-2018_TrafficForML_CICFlowMeter.csv,Benign,2,100.000000
7,Thursday-01-03-2018_TrafficForML_CICFlowMeter.csv,Benign,50,68.493151
8,Thursday-01-03-2018_TrafficForML_CICFlowMeter.csv,Infilteration,23,31.506849
9,Thursday-15-02-2018_TrafficForML_CICFlowMeter.csv,Benign,1663,68.690624



OVERALL DUPLICATE RESULTS
Total rows investigated: 16,232,929
Total exact duplicate rows: 410,707
Overall duplicate percentage: 2.5301%

No duplicate records were removed in this step.


## 2.6 — Exact Duplicate Removal

The duplicate investigation identified 410,707 exact duplicate records,
representing approximately 2.53% of the dataset.


Because these records are identical across all available columns, retaining
all copies could give repeated observations disproportionate influence during
model training and temporal state construction.


Therefore, exact duplicate rows are removed while retaining the first
occurrence of each unique record.


Only exact duplicates are removed. Similar or near-duplicate network flows
are retained.

In [16]:
# ============================================================
# PHASE 2 — DATA PREPROCESSING
# 2.6 — EXACT DUPLICATE REMOVAL
# ============================================================

import os
import duckdb
import pandas as pd

# ------------------------------------------------------------
# Configuration
# ------------------------------------------------------------

INPUT_DIR = "/content/phase2_quality_cleaned"
OUTPUT_DIR = "/content/phase2_deduplicated"

os.makedirs(OUTPUT_DIR, exist_ok=True)

print("=" * 70)
print("PHASE 2.6 — EXACT DUPLICATE REMOVAL")
print("=" * 70)

dedup_results = []

# ------------------------------------------------------------
# Process each file
# ------------------------------------------------------------

for file_number, file_name in enumerate(files_to_process, start=1):

    input_path = os.path.join(INPUT_DIR, file_name)
    output_path = os.path.join(OUTPUT_DIR, file_name)

    print(f"\n[{file_number}/10] Processing:")
    print(file_name)

    if os.path.exists(output_path):
        os.remove(output_path)

    con = duckdb.connect()

    try:

        # ----------------------------------------------------
        # Remove exact duplicate rows.
        #
        # DuckDB DISTINCT keeps one copy of identical rows.
        # ----------------------------------------------------

        query = f"""
        COPY (
            SELECT DISTINCT *
            FROM read_csv_auto(
                '{input_path}',
                header = true,
                ignore_errors = false
            )
        )
        TO '{output_path}'
        WITH (
            HEADER,
            DELIMITER ','
        );
        """

        con.execute(query)

        # ----------------------------------------------------
        # Count input rows
        # ----------------------------------------------------

        input_count = con.execute(
            f"""
            SELECT COUNT(*)
            FROM read_csv_auto(
                '{input_path}',
                header = true
            );
            """
        ).fetchone()[0]

        # ----------------------------------------------------
        # Count output rows
        # ----------------------------------------------------

        output_count = con.execute(
            f"""
            SELECT COUNT(*)
            FROM read_csv_auto(
                '{output_path}',
                header = true
            );
            """
        ).fetchone()[0]

    finally:
        con.close()

    # --------------------------------------------------------
    # Calculate duplicate removal
    # --------------------------------------------------------

    duplicates_removed = input_count - output_count

    duplicate_percentage = (
        duplicates_removed / input_count * 100
        if input_count > 0
        else 0
    )

    dedup_results.append({
        "File": file_name,
        "Input Rows": input_count,
        "Unique Rows": output_count,
        "Duplicates Removed": duplicates_removed,
        "Duplicate Percentage": duplicate_percentage
    })

    print(f"  Input rows          : {input_count:,}")
    print(f"  Unique rows         : {output_count:,}")
    print(f"  Duplicates removed  : {duplicates_removed:,}")
    print(f"  Percentage removed  : {duplicate_percentage:.4f}%")

# ------------------------------------------------------------
# Summary
# ------------------------------------------------------------

dedup_df = pd.DataFrame(dedup_results)

print("\n" + "=" * 70)
print("DUPLICATE REMOVAL SUMMARY")
print("=" * 70)

display(dedup_df)

# ------------------------------------------------------------
# Overall statistics
# ------------------------------------------------------------

total_input = dedup_df["Input Rows"].sum()
total_output = dedup_df["Unique Rows"].sum()
total_removed = dedup_df["Duplicates Removed"].sum()

overall_percentage = (
    total_removed / total_input * 100
)

print("\n" + "=" * 70)
print("OVERALL RESULTS")
print("=" * 70)

print(f"Total input rows       : {total_input:,}")
print(f"Total unique rows      : {total_output:,}")
print(f"Total duplicates removed: {total_removed:,}")
print(f"Overall removed        : {overall_percentage:.4f}%")

print("\nDeduplicated files saved in:")
print(OUTPUT_DIR)

print("\nOnly exact duplicate rows were removed.")

PHASE 2.6 — EXACT DUPLICATE REMOVAL

[1/10] Processing:
Friday-02-03-2018_TrafficForML_CICFlowMeter.csv


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

  Input rows          : 1,048,575
  Unique rows         : 1,043,116
  Duplicates removed  : 5,459
  Percentage removed  : 0.5206%

[2/10] Processing:
Friday-16-02-2018_TrafficForML_CICFlowMeter.csv


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

  Input rows          : 1,048,574
  Unique rows         : 900,988
  Duplicates removed  : 147,586
  Percentage removed  : 14.0749%

[3/10] Processing:
Friday-23-02-2018_TrafficForML_CICFlowMeter.csv


... (24 outputs left out)

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

  Input rows          : 613,071
  Unique rows         : 606,982
  Duplicates removed  : 6,089
  Percentage removed  : 0.9932%

DUPLICATE REMOVAL SUMMARY


,File,Input Rows,Unique Rows,Duplicates Removed,Duplicate Percentage
0,Friday-02-03-2018_TrafficForML_CICFlowMeter.csv,1048575,1043116,5459,0.520611
1,Friday-16-02-2018_TrafficForML_CICFlowMeter.csv,1048574,900988,147586,14.074925
2,Friday-23-02-2018_TrafficForML_CICFlowMeter.csv,1048575,1045961,2614,0.249291
3,Thuesday-20-02-2018_TrafficForML_CICFlowMeter.csv,7948748,7948746,2,0.000025
4,Thursday-01-03-2018_TrafficForML_CICFlowMeter.csv,331100,331027,73,0.022048
5,Thursday-15-02-2018_TrafficForML_CICFlowMeter.csv,1048575,1046154,2421,0.230885
6,Thursday-22-02-2018_TrafficForML_CICFlowMeter.csv,1048566,1045288,3278,0.312617
7,Wednesday-14-02-2018_TrafficForML_CICFlowMeter...,1048570,822942,225628,21.517686
8,Wednesday-21-02-2018_TrafficForML_CICFlowMeter...,1048575,1031018,17557,1.674368
9,Wednesday-28-02-2018_TrafficForML_CICFlowMeter...,613071,606982,6089,0.993197



OVERALL RESULTS
Total input rows       : 16,232,929
Total unique rows      : 15,822,222
Total duplicates removed: 410,707
Overall removed        : 2.5301%

Deduplicated files saved in:
/content/phase2_deduplicated

Only exact duplicate rows were removed.


## 2.7 — Negative & Invalid Numerical Value Investigation

Network-flow features are expected to represent quantities such as packet
counts, byte counts, durations, rates, and inter-arrival times.

Phase 1 identified negative values in several numerical features. However,
negative values cannot be removed indiscriminately because some CICFlowMeter
features may use `-1` as a sentinel value to represent unavailable or
undefined information.

This step therefore investigates negative values feature-by-feature.

The investigation records:

- Number of negative values
- Number of `-1` values
- Minimum value
- Affected files

No negative values are modified or removed during this investigation.

In [17]:
# ============================================================
# PHASE 2 — DATA PREPROCESSING
# 2.7 — NEGATIVE & INVALID NUMERICAL VALUE INVESTIGATION
# ============================================================

import os
import numpy as np
import pandas as pd

# ------------------------------------------------------------
# Configuration
# ------------------------------------------------------------

INPUT_DIR = "/content/phase2_deduplicated"

CHUNK_SIZE = 100_000

print("=" * 70)
print("PHASE 2.7 — NEGATIVE & INVALID NUMERICAL VALUE INVESTIGATION")
print("=" * 70)

negative_summary = []

# ------------------------------------------------------------
# Process all files
# ------------------------------------------------------------

for file_number, file_name in enumerate(files_to_process, start=1):

    file_path = os.path.join(INPUT_DIR, file_name)

    print(f"\n[{file_number}/10] Investigating:")
    print(file_name)

    # Feature-level accumulators
    feature_stats = {}

    # --------------------------------------------------------
    # Read file in chunks
    # --------------------------------------------------------

    for chunk in pd.read_csv(
        file_path,
        chunksize=CHUNK_SIZE,
        low_memory=False
    ):

        # ----------------------------------------------------
        # Identify numerical columns
        # ----------------------------------------------------

        numeric_columns = chunk.select_dtypes(
            include=[np.number]
        ).columns

        for column in numeric_columns:

            series = chunk[column]

            # Ignore missing values
            valid_values = series.dropna()

            if valid_values.empty:
                continue

            # ------------------------------------------------
            # Negative values
            # ------------------------------------------------

            negative_count = int(
                (valid_values < 0).sum()
            )

            if negative_count == 0:
                continue

            # ------------------------------------------------
            # -1 values
            # ------------------------------------------------

            minus_one_count = int(
                (valid_values == -1).sum()
            )

            # ------------------------------------------------
            # Minimum value
            # ------------------------------------------------

            minimum_value = valid_values.min()

            # ------------------------------------------------
            # Other negative values excluding -1
            # ------------------------------------------------

            other_negative_count = int(
                (
                    (valid_values < 0)
                    & (valid_values != -1)
                ).sum()
            )

            # ------------------------------------------------
            # Store accumulated statistics
            # ------------------------------------------------

            if column not in feature_stats:

                feature_stats[column] = {
                    "Negative Values": 0,
                    "-1 Values": 0,
                    "Other Negative Values": 0,
                    "Minimum Value": minimum_value
                }

            feature_stats[column]["Negative Values"] += (
                negative_count
            )

            feature_stats[column]["-1 Values"] += (
                minus_one_count
            )

            feature_stats[column]["Other Negative Values"] += (
                other_negative_count
            )

            feature_stats[column]["Minimum Value"] = min(
                feature_stats[column]["Minimum Value"],
                minimum_value
            )

    # --------------------------------------------------------
    # Convert file statistics into rows
    # --------------------------------------------------------

    for feature, stats in feature_stats.items():

        negative_summary.append({
            "File": file_name,
            "Feature": feature,
            "Negative Values": stats["Negative Values"],
            "-1 Values": stats["-1 Values"],
            "Other Negative Values":
                stats["Other Negative Values"],
            "Minimum Value": stats["Minimum Value"]
        })

# ------------------------------------------------------------
# Create summary DataFrame
# ------------------------------------------------------------

negative_df = pd.DataFrame(
    negative_summary
)

print("\n" + "=" * 70)
print("NEGATIVE VALUE SUMMARY")
print("=" * 70)

if not negative_df.empty:

    display(
        negative_df.sort_values(
            ["File", "Negative Values"],
            ascending=[True, False]
        )
    )

else:

    print("No negative numerical values detected.")

# ------------------------------------------------------------
# Aggregate by feature
# ------------------------------------------------------------

if not negative_df.empty:

    feature_totals = (
        negative_df
        .groupby("Feature", as_index=False)
        .agg({
            "Negative Values": "sum",
            "-1 Values": "sum",
            "Other Negative Values": "sum",
            "Minimum Value": "min"
        })
        .sort_values(
            "Negative Values",
            ascending=False
        )
    )

    print("\n" + "=" * 70)
    print("AGGREGATED NEGATIVE VALUES BY FEATURE")
    print("=" * 70)

    display(feature_totals)

# ------------------------------------------------------------
# Important classification
# ------------------------------------------------------------

print("\n" + "=" * 70)
print("INVESTIGATION STATUS")
print("=" * 70)

print("- Negative values have NOT been removed.")
print("- -1 values have NOT been converted.")
print("- No assumptions about sentinel values have been made.")
print("- The feature-level results will determine the next action.")

PHASE 2.7 — NEGATIVE & INVALID NUMERICAL VALUE INVESTIGATION

[1/10] Investigating:
Friday-02-03-2018_TrafficForML_CICFlowMeter.csv

[2/10] Investigating:
Friday-16-02-2018_TrafficForML_CICFlowMeter.csv

[3/10] Investigating:
Friday-23-02-2018_TrafficForML_CICFlowMeter.csv

[4/10] Investigating:
Thuesday-20-02-2018_TrafficForML_CICFlowMeter.csv

[5/10] Investigating:
Thursday-01-03-2018_TrafficForML_CICFlowMeter.csv

[6/10] Investigating:
Thursday-15-02-2018_TrafficForML_CICFlowMeter.csv

[7/10] Investigating:
Thursday-22-02-2018_TrafficForML_CICFlowMeter.csv

[8/10] Investigating:
Wednesday-14-02-2018_TrafficForML_CICFlowMeter.csv

[9/10] Investigating:
Wednesday-21-02-2018_TrafficForML_CICFlowMeter.csv

[10/10] Investigating:
Wednesday-28-02-2018_TrafficForML_CICFlowMeter.csv

NEGATIVE VALUE SUMMARY


,File,Feature,Negative Values,-1 Values,Other Negative Values,Minimum Value
1,Friday-02-03-2018_TrafficForML_CICFlowMeter.csv,Init Bwd Win Byts,508990,508990,0,-1
0,Friday-02-03-2018_TrafficForML_CICFlowMeter.csv,Init Fwd Win Byts,216035,216035,0,-1
3,Friday-16-02-2018_TrafficForML_CICFlowMeter.csv,Init Bwd Win Byts,411239,411239,0,-1
2,Friday-16-02-2018_TrafficForML_CICFlowMeter.csv,Init Fwd Win Byts,125,125,0,-1
5,Friday-23-02-2018_TrafficForML_CICFlowMeter.csv,Init Bwd Win Byts,570603,570603,0,-1
4,Friday-23-02-2018_TrafficForML_CICFlowMeter.csv,Init Fwd Win Byts,308367,308367,0,-1
6,Friday-23-02-2018_TrafficForML_CICFlowMeter.csv,Flow IAT Min,1,0,1,-2228
7,Friday-23-02-2018_TrafficForML_CICFlowMeter.csv,Fwd IAT Min,1,0,1,-2171
9,Thuesday-20-02-2018_TrafficForML_CICFlowMeter.csv,Init Bwd Win Byts,4209793,4209793,0,-1
8,Thuesday-20-02-2018_TrafficForML_CICFlowMeter.csv,Init Fwd Win Byts,2653104,2653104,0,-1



AGGREGATED NEGATIVE VALUES BY FEATURE


,Feature,Negative Values,-1 Values,Other Negative Values,Minimum Value
2,Init Bwd Win Byts,8192276,8192276,0,-1
3,Init Fwd Win Byts,4432096,4432096,0,-1
1,Fwd IAT Min,1,0,1,-2171
0,Flow IAT Min,1,0,1,-2228



INVESTIGATION STATUS
- Negative values have NOT been removed.
- -1 values have NOT been converted.
- No assumptions about sentinel values have been made.
- The feature-level results will determine the next action.


## 2.7.1 — Locate Invalid Negative Records

Phase 2.7 identified two physically invalid negative values:

- `Flow IAT Min = -2228`
- `Fwd IAT Min = -2171`

Unlike the `-1` sentinel values found in the initial TCP window features,
these values are not treated as valid sentinel values.

This step locates the affected records and determines whether the two
invalid values belong to the same network-flow record or to separate records.

No records are removed during this investigation.

In [18]:
# ============================================================
# PHASE 2 — DATA PREPROCESSING
# 2.7.1 — LOCATE INVALID NEGATIVE RECORDS
# ============================================================

import os
import pandas as pd

INPUT_DIR = "/content/phase2_deduplicated"

TARGET_FEATURES = [
    "Flow IAT Min",
    "Fwd IAT Min"
]

print("=" * 70)
print("PHASE 2.7.1 — LOCATING INVALID NEGATIVE RECORDS")
print("=" * 70)

invalid_records = []

for file_number, file_name in enumerate(files_to_process, start=1):

    file_path = os.path.join(INPUT_DIR, file_name)

    for chunk in pd.read_csv(
        file_path,
        chunksize=100_000,
        low_memory=False
    ):

        # ----------------------------------------------------
        # Find values below zero excluding the valid -1
        # sentinel values.
        # ----------------------------------------------------

        invalid_mask = (
            (chunk["Flow IAT Min"] < 0)
            | (chunk["Fwd IAT Min"] < 0)
        )

        invalid_chunk = chunk.loc[
            invalid_mask
        ].copy()

        if invalid_chunk.empty:
            continue

        # ----------------------------------------------------
        # Keep only relevant information for investigation
        # ----------------------------------------------------

        columns_to_show = [
            "Timestamp",
            "Label",
            "Flow Duration",
            "Tot Fwd Pkts",
            "Tot Bwd Pkts",
            "Flow IAT Min",
            "Fwd IAT Min",
            "Flow Pkts/s",
            "Flow Byts/s"
        ]

        columns_to_show = [
            col for col in columns_to_show
            if col in invalid_chunk.columns
        ]

        records = invalid_chunk[columns_to_show].copy()

        records.insert(
            0,
            "File",
            file_name
        )

        invalid_records.append(records)

# ------------------------------------------------------------
# Display results
# ------------------------------------------------------------

print("\n" + "=" * 70)
print("INVALID NEGATIVE RECORDS FOUND")
print("=" * 70)

if invalid_records:

    invalid_df = pd.concat(
        invalid_records,
        ignore_index=True
    )

    display(invalid_df)

    print("\nNumber of affected records:", len(invalid_df))

    # Check whether both invalid features occur in the same row
    both_invalid = (
        (invalid_df["Flow IAT Min"] < 0)
        & (invalid_df["Fwd IAT Min"] < 0)
    ).sum()

    flow_iat_invalid = (
        invalid_df["Flow IAT Min"] < 0
    ).sum()

    fwd_iat_invalid = (
        invalid_df["Fwd IAT Min"] < 0
    ).sum()

    print(
        "Records with Flow IAT Min < 0:",
        flow_iat_invalid
    )

    print(
        "Records with Fwd IAT Min < 0:",
        fwd_iat_invalid
    )

    print(
        "Records where BOTH are negative:",
        both_invalid
    )

else:

    print("No invalid negative records found.")

print("\nNo records were removed in this investigation.")

PHASE 2.7.1 — LOCATING INVALID NEGATIVE RECORDS

INVALID NEGATIVE RECORDS FOUND


,File,Timestamp,Label,Flow Duration,Tot Fwd Pkts,Tot Bwd Pkts,Flow IAT Min,Fwd IAT Min,Flow Pkts/s,Flow Byts/s
0,Friday-23-02-2018_TrafficForML_CICFlowMeter.csv,2018-02-23 10:22:22,Benign,5132268,10,8,-2228,-2171,3.507221,393.003639



Number of affected records: 1
Records with Flow IAT Min < 0: 1
Records with Fwd IAT Min < 0: 1
Records where BOTH are negative: 1

No records were removed in this investigation.


In [19]:
# ================================================================
# PHASE 2.7.2 — REMOVE INVALID NEGATIVE RECORDS
# ================================================================
# This cell:
# 1. Reads the deduplicated Phase 2 dataset.
# 2. Removes only records with genuinely invalid negative IAT values.
# 3. Preserves -1 values in Init Fwd/Bwd Win Byts.
# 4. Saves the cleaned files for the next preprocessing step.
# ================================================================

import os
import glob
import duckdb
import pandas as pd

INPUT_DIR = "/content/phase2_deduplicated"
OUTPUT_DIR = "/content/phase2_numeric_cleaned"

os.makedirs(OUTPUT_DIR, exist_ok=True)

csv_files = sorted(glob.glob(os.path.join(INPUT_DIR, "*.csv")))

print("=" * 70)
print("PHASE 2.7.2 — REMOVING INVALID NEGATIVE RECORDS")
print("=" * 70)

total_input = 0
total_removed = 0
total_output = 0

results = []

for input_file in csv_files:

    filename = os.path.basename(input_file)
    output_file = os.path.join(OUTPUT_DIR, filename)

    con = duckdb.connect()

    # Count input rows
    input_count = con.execute(f"""
        SELECT COUNT(*)
        FROM read_csv_auto(
            '{input_file}',
            header=True,
            ignore_errors=True
        )
    """).fetchone()[0]

    # Count genuinely invalid negative records.
    # -1 values in Init Fwd/Bwd Win Byts are NOT included.
    invalid_count = con.execute(f"""
        SELECT COUNT(*)
        FROM read_csv_auto(
            '{input_file}',
            header=True,
            ignore_errors=True
        )
        WHERE
            TRY_CAST("Flow IAT Min" AS DOUBLE) < 0
            OR
            TRY_CAST("Fwd IAT Min" AS DOUBLE) < 0
    """).fetchone()[0]

    # Remove only the invalid IAT records.
    con.execute(f"""
        COPY (
            SELECT *
            FROM read_csv_auto(
                '{input_file}',
                header=True,
                ignore_errors=True
            )
            WHERE
                NOT (
                    TRY_CAST("Flow IAT Min" AS DOUBLE) < 0
                    OR
                    TRY_CAST("Fwd IAT Min" AS DOUBLE) < 0
                )
        )
        TO '{output_file}'
        (HEADER, DELIMITER ',')
    """)

    output_count = con.execute(f"""
        SELECT COUNT(*)
        FROM read_csv_auto(
            '{output_file}',
            header=True,
            ignore_errors=True
        )
    """).fetchone()[0]

    con.close()

    total_input += input_count
    total_removed += invalid_count
    total_output += output_count

    results.append({
        "File": filename,
        "Input Rows": input_count,
        "Invalid Negative Rows Removed": invalid_count,
        "Output Rows": output_count
    })

    print(f"{filename}")
    print(f"  Input rows   : {input_count:,}")
    print(f"  Removed      : {invalid_count:,}")
    print(f"  Output rows  : {output_count:,}")
    print()

print("=" * 70)
print("PHASE 2.7.2 SUMMARY")
print("=" * 70)

print(f"Total input rows   : {total_input:,}")
print(f"Total removed      : {total_removed:,}")
print(f"Total output rows  : {total_output:,}")

print()
print("Output directory:")
print(OUTPUT_DIR)

print()
print("Invalid negative records have been removed.")
print("Allowed -1 values in Init Fwd/Bwd Win Byts were preserved.")


PHASE 2.7.2 — REMOVING INVALID NEGATIVE RECORDS


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Friday-02-03-2018_TrafficForML_CICFlowMeter.csv
  Input rows   : 1,043,116
  Removed      : 0
  Output rows  : 1,043,116



FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Friday-16-02-2018_TrafficForML_CICFlowMeter.csv
  Input rows   : 900,988
  Removed      : 0
  Output rows  : 900,988



... (20 outputs left out)

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Wednesday-21-02-2018_TrafficForML_CICFlowMeter.csv
  Input rows   : 1,031,018
  Removed      : 0
  Output rows  : 1,031,018



FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Wednesday-28-02-2018_TrafficForML_CICFlowMeter.csv
  Input rows   : 606,982
  Removed      : 0
  Output rows  : 606,982

PHASE 2.7.2 SUMMARY
Total input rows   : 15,822,222
Total removed      : 1
Total output rows  : 15,822,221

Output directory:
/content/phase2_numeric_cleaned

Invalid negative records have been removed.
Allowed -1 values in Init Fwd/Bwd Win Byts were preserved.


##Phase 2.8 — Schema & Data-Type Normalization

Now we verify that the cleaned dataset has a usable and consistent schema before doing feature engineering.

What this step will check

For all 10 processed files, we will verify:

1.Number of columns

2.Column names

3.Timestamp presence

4.Label presence

5.Numeric vs non-numeric columns


Whether numeric traffic features are actually numeric

The Tuesday file's additional 4 columns
Any remaining unexpected schema/type problems

We will not drop any columns yet. This is normalization/investigation only.

In [20]:
# ================================================================
# PHASE 2.8.1 — SCHEMA & DATA-TYPE VALIDATION
# ================================================================
# This cell:
# 1. Reads all 10 cleaned CSV files.
# 2. Checks column count and column names.
# 3. Identifies numeric and non-numeric columns.
# 4. Verifies Timestamp and Label.
# 5. Checks whether expected numeric features are truly numeric.
# 6. Identifies schema differences between files.
# 7. Does NOT remove or modify any columns.
# ================================================================

import os
import glob
import pandas as pd
import numpy as np

INPUT_DIR = "/content/phase2_numeric_cleaned"

csv_files = sorted(glob.glob(os.path.join(INPUT_DIR, "*.csv")))

print("=" * 75)
print("PHASE 2.8.1 — SCHEMA & DATA-TYPE VALIDATION")
print("=" * 75)

print(f"\nFiles found: {len(csv_files)}")

# ------------------------------------------------
# 1. Use the first file as the reference schema
# ------------------------------------------------

reference_file = csv_files[0]

reference_df = pd.read_csv(
    reference_file,
    nrows=1000,
    low_memory=False
)

reference_columns = list(reference_df.columns)

print("\n" + "=" * 75)
print("REFERENCE SCHEMA")
print("=" * 75)

print(f"Reference file : {os.path.basename(reference_file)}")
print(f"Reference columns: {len(reference_columns)}")

# ------------------------------------------------
# 2. Define expected special columns
# ------------------------------------------------

special_columns = {
    "Timestamp",
    "Label"
}

# Tuesday-specific endpoint/identifier columns
tuesday_extra_columns = {
    "Flow ID",
    "Src IP",
    "Src Port",
    "Dst IP"
}

# ------------------------------------------------
# 3. Inspect every file
# ------------------------------------------------

schema_results = []

all_schema_differences = []

for file_path in csv_files:

    filename = os.path.basename(file_path)

    # Read only a sample for schema/type inspection
    df = pd.read_csv(
        file_path,
        nrows=5000,
        low_memory=False
    )

    columns = list(df.columns)

    numeric_columns = df.select_dtypes(
        include=[np.number]
    ).columns.tolist()

    non_numeric_columns = [
        col for col in columns
        if col not in numeric_columns
    ]

    missing_timestamp = "Timestamp" not in columns
    missing_label = "Label" not in columns

    missing_reference_columns = [
        col for col in reference_columns
        if col not in columns
    ]

    extra_columns = [
        col for col in columns
        if col not in reference_columns
    ]

    schema_same = (
        columns == reference_columns
    )

    schema_results.append({
        "File": filename,
        "Columns": len(columns),
        "Numeric Columns": len(numeric_columns),
        "Non-Numeric Columns": len(non_numeric_columns),
        "Timestamp": not missing_timestamp,
        "Label": not missing_label,
        "Schema Same as Reference": schema_same,
        "Extra Columns": ", ".join(extra_columns) if extra_columns else "None",
        "Missing Reference Columns": (
            ", ".join(missing_reference_columns)
            if missing_reference_columns else "None"
        )
    })

    if not schema_same:

        all_schema_differences.append({
            "File": filename,
            "Missing Columns": missing_reference_columns,
            "Extra Columns": extra_columns
        })

    # ------------------------------------------------
    # Print file-level result
    # ------------------------------------------------

    print("\n" + "-" * 75)
    print(filename)
    print("-" * 75)

    print(f"Columns           : {len(columns)}")
    print(f"Numeric columns   : {len(numeric_columns)}")
    print(f"Non-numeric cols  : {len(non_numeric_columns)}")
    print(f"Timestamp present : {'YES' if not missing_timestamp else 'NO'}")
    print(f"Label present     : {'YES' if not missing_label else 'NO'}")
    print(
        f"Schema matches reference : "
        f"{'YES' if schema_same else 'NO'}"
    )

    if extra_columns:
        print(f"Extra columns     : {extra_columns}")

    if missing_reference_columns:
        print(f"Missing columns   : {missing_reference_columns}")

    print(f"Non-numeric columns: {non_numeric_columns}")

# ------------------------------------------------
# 4. Create summary DataFrame
# ------------------------------------------------

summary_df = pd.DataFrame(schema_results)

print("\n" + "=" * 75)
print("SCHEMA SUMMARY")
print("=" * 75)

display(summary_df)

# ------------------------------------------------
# 5. Check the expected Tuesday-specific columns
# ------------------------------------------------

print("\n" + "=" * 75)
print("TUESDAY-SPECIFIC COLUMN CHECK")
print("=" * 75)

tuesday_files = [
    f for f in csv_files
    if "Thuesday-20-02-2018" in os.path.basename(f)
]

if tuesday_files:

    tuesday_df = pd.read_csv(
        tuesday_files[0],
        nrows=1000,
        low_memory=False
    )

    present_tuesday_extra = [
        col for col in tuesday_extra_columns
        if col in tuesday_df.columns
    ]

    print("Expected Tuesday-specific columns:")
    print(sorted(tuesday_extra_columns))

    print("\nPresent:")
    print(sorted(present_tuesday_extra))

    print(
        "\nAll four Tuesday-specific columns present:",
        len(present_tuesday_extra) == 4
    )

# ------------------------------------------------
# 6. Check datatype consistency of common columns
# ------------------------------------------------

print("\n" + "=" * 75)
print("COMMON COLUMN DATA-TYPE CHECK")
print("=" * 75)

dtype_issues = []

common_columns = set(reference_columns)

for file_path in csv_files:

    filename = os.path.basename(file_path)

    df = pd.read_csv(
        file_path,
        nrows=5000,
        low_memory=False
    )

    for column in common_columns:

        if column not in df.columns:
            continue

        dtype_name = str(df[column].dtype)

        # Timestamp and Label are expected to be non-numeric
        if column in special_columns:
            continue

        # Tuesday IP/Flow ID fields are identifiers
        if column in tuesday_extra_columns:
            continue

        # All other common traffic features should be numeric
        if not pd.api.types.is_numeric_dtype(df[column]):

            dtype_issues.append({
                "File": filename,
                "Column": column,
                "Detected dtype": dtype_name
            })

if dtype_issues:

    print("Potential numeric dtype issues found:")
    display(pd.DataFrame(dtype_issues))

else:

    print(
        "No unexpected non-numeric traffic feature types "
        "found in the inspected samples."
    )

# ------------------------------------------------
# 7. Final status
# ------------------------------------------------

print("\n" + "=" * 75)
print("PHASE 2.8.1 STATUS")
print("=" * 75)

print(f"Files inspected                 : {len(csv_files)}")
print(
    f"Files with schema differences   : "
    f"{len(all_schema_differences)}"
)
print(
    f"Unexpected numeric dtype issues : "
    f"{len(dtype_issues)}"
)

print("\nNo columns were removed or modified.")
print("Schema investigation completed.")

PHASE 2.8.1 — SCHEMA & DATA-TYPE VALIDATION

Files found: 10

REFERENCE SCHEMA
Reference file : Friday-02-03-2018_TrafficForML_CICFlowMeter.csv
Reference columns: 80

---------------------------------------------------------------------------
Friday-02-03-2018_TrafficForML_CICFlowMeter.csv
---------------------------------------------------------------------------
Columns           : 80
Numeric columns   : 78
Non-numeric cols  : 2
Timestamp present : YES
Label present     : YES
Schema matches reference : YES
Non-numeric columns: ['Timestamp', 'Label']

---------------------------------------------------------------------------
Friday-16-02-2018_TrafficForML_CICFlowMeter.csv
---------------------------------------------------------------------------
Columns           : 80
Numeric columns   : 78
Non-numeric cols  : 2
Timestamp present : YES
Label present     : YES
Schema matches reference : YES
Non-numeric columns: ['Timestamp', 'Label']

-------------------------------------------------

,File,Columns,Numeric Columns,Non-Numeric Columns,Timestamp,Label,Schema Same as Reference,Extra Columns,Missing Reference Columns
0,Friday-02-03-2018_TrafficForML_CICFlowMeter.csv,80,78,2,True,True,True,None,None
1,Friday-16-02-2018_TrafficForML_CICFlowMeter.csv,80,78,2,True,True,True,None,None
2,Friday-23-02-2018_TrafficForML_CICFlowMeter.csv,80,78,2,True,True,True,None,None
3,Thuesday-20-02-2018_TrafficForML_CICFlowMeter.csv,84,79,5,True,True,False,"Flow ID, Src IP, Src Port, Dst IP",None
4,Thursday-01-03-2018_TrafficForML_CICFlowMeter.csv,80,78,2,True,True,True,None,None
5,Thursday-15-02-2018_TrafficForML_CICFlowMeter.csv,80,78,2,True,True,True,None,None
6,Thursday-22-02-2018_TrafficForML_CICFlowMeter.csv,80,78,2,True,True,True,None,None
7,Wednesday-14-02-2018_TrafficForML_CICFlowMeter...,80,78,2,True,True,True,None,None
8,Wednesday-21-02-2018_TrafficForML_CICFlowMeter...,80,78,2,True,True,True,None,None
9,Wednesday-28-02-2018_TrafficForML_CICFlowMeter...,80,78,2,True,True,True,None,None



TUESDAY-SPECIFIC COLUMN CHECK
Expected Tuesday-specific columns:
['Dst IP', 'Flow ID', 'Src IP', 'Src Port']

Present:
['Dst IP', 'Flow ID', 'Src IP', 'Src Port']

All four Tuesday-specific columns present: True

COMMON COLUMN DATA-TYPE CHECK
No unexpected non-numeric traffic feature types found in the inspected samples.

PHASE 2.8.1 STATUS
Files inspected                 : 10
Files with schema differences   : 1
Unexpected numeric dtype issues : 0

No columns were removed or modified.
Schema investigation completed.


##Phase 2.8.2 — Normalize Numeric Data Types
Now that the schema is understood, we can safely normalize the actual traffic-feature types.

The goal is not feature selection yet. We simply make sure the common numerical columns are consistently represented as numeric values and preserve:

Timestamp

Label

Tuesday's Flow ID

Src IP

Dst IP


We also need to retain the NaN values currently present in Flow Byts/s and Flow Pkts/s. Do not fill them globally yet; we'll handle those appropriately later to avoid data leakage.

In [1]:
# ================================================================
# PHASE 2.8.2 — MEMORY-EFFICIENT NUMERIC TYPE VALIDATION
# ================================================================
# This cell:
# 1. Reads each cleaned CSV using DuckDB.
# 2. Checks all expected numeric traffic columns.
# 3. Counts values that cannot be interpreted as numeric.
# 4. Does NOT load the complete dataset into Pandas.
# 5. Does NOT rewrite the 6.9 GB dataset.
#
# This is intentionally validation-only.
# Actual ML-time dtype conversion will be performed later
# when constructing the feature matrix.
# ================================================================

import os
import glob
import duckdb
import pandas as pd

INPUT_DIR = "/content/phase2_numeric_cleaned"

csv_files = sorted(
    glob.glob(os.path.join(INPUT_DIR, "*.csv"))
)

# Columns that are intentionally non-numeric
NON_NUMERIC_COLUMNS = {
    "Timestamp",
    "Label",
    "Flow ID",
    "Src IP",
    "Dst IP"
}

print("=" * 75)
print("PHASE 2.8.2 — MEMORY-EFFICIENT NUMERIC TYPE VALIDATION")
print("=" * 75)

print(f"\nFiles to validate: {len(csv_files)}")

validation_results = []

total_unconvertible = 0

for index, file_path in enumerate(csv_files, start=1):

    filename = os.path.basename(file_path)

    print("\n" + "-" * 75)
    print(f"[{index}/{len(csv_files)}] {filename}")
    print("-" * 75)

    con = duckdb.connect()

    # ------------------------------------------------------------
    # Read only the header to identify columns
    # ------------------------------------------------------------

    columns_df = con.execute(f"""
        DESCRIBE
        SELECT *
        FROM read_csv_auto(
            '{file_path}',
            header=True
        )
    """).fetchdf()

    all_columns = columns_df["column_name"].tolist()

    numeric_columns = [
        col for col in all_columns
        if col not in NON_NUMERIC_COLUMNS
    ]

    file_unconvertible = 0

    # ------------------------------------------------------------
    # Check each numeric feature
    # ------------------------------------------------------------

    for col in numeric_columns:

        result = con.execute(f"""
            SELECT COUNT(*)
            FROM read_csv_auto(
                '{file_path}',
                header=True,
                ignore_errors=True
            )
            WHERE
                "{col}" IS NOT NULL
                AND TRY_CAST("{col}" AS DOUBLE) IS NULL
        """).fetchone()

        invalid_count = result[0]

        if invalid_count > 0:

            print(
                f"  WARNING: {col} -> "
                f"{invalid_count:,} non-numeric values"
            )

            file_unconvertible += invalid_count

    # ------------------------------------------------------------
    # File result
    # ------------------------------------------------------------

    total_unconvertible += file_unconvertible

    validation_results.append({
        "File": filename,
        "Columns": len(all_columns),
        "Numeric Traffic Features": len(numeric_columns),
        "Unconvertible Numeric Values": file_unconvertible
    })

    if file_unconvertible == 0:
        print("  Numeric validation: PASS")
    else:
        print("  Numeric validation: REVIEW REQUIRED")

    con.close()

# ================================================================
# FINAL SUMMARY
# ================================================================

results_df = pd.DataFrame(validation_results)

print("\n" + "=" * 75)
print("PHASE 2.8.2 SUMMARY")
print("=" * 75)

display(results_df)

print("\nTotal unconvertible numeric values:",
      f"{total_unconvertible:,}")

if total_unconvertible == 0:

    print("\nSTATUS: PASS")
    print(
        "All traffic feature values can be interpreted "
        "as numeric."
    )

else:

    print("\nSTATUS: REVIEW REQUIRED")
    print(
        "Some traffic feature values could not be "
        "interpreted as numeric."
    )

print("\nNo large Pandas dataframe was created.")
print("No files were rewritten.")
print("Phase 2.8.2 validation completed.")

PHASE 2.8.2 — MEMORY-EFFICIENT NUMERIC TYPE VALIDATION

Files to validate: 10

---------------------------------------------------------------------------
[1/10] Friday-02-03-2018_TrafficForML_CICFlowMeter.csv
---------------------------------------------------------------------------
  Numeric validation: PASS

---------------------------------------------------------------------------
[2/10] Friday-16-02-2018_TrafficForML_CICFlowMeter.csv
---------------------------------------------------------------------------


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

  Numeric validation: PASS

---------------------------------------------------------------------------
[3/10] Friday-23-02-2018_TrafficForML_CICFlowMeter.csv
---------------------------------------------------------------------------


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

  Numeric validation: PASS

---------------------------------------------------------------------------
[4/10] Thuesday-20-02-2018_TrafficForML_CICFlowMeter.csv
---------------------------------------------------------------------------


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

... (65 outputs left out)

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

KeyboardInterrupt: 

##Phase 2.9 — Missing-Value Treatment for Derived Rate Features

Now we address the remaining missing values from Phase 2.5.

We know that missing/infinite values occurred only in:

Flow Byts/s

Flow Pkts/s


We will not immediately impute them. First, we need to determine whether these values can be reconstructed from the underlying traffic measurements.

What this investigation will check

For affected records, we'll examine:

Flow Duration
Tot Fwd Pkts
Tot Bwd Pkts
TotLen Fwd Pkts
TotLen Bwd Pkts
Flow Byts/s
Flow Pkts/s

The purpose is to determine whether the missing rates correspond to mathematically reconstructable values or genuinely undefined rates.

Because the dataset is very large, we'll inspect a representative sample of affected records, rather than scanning every row repeatedly.

In [2]:
# ================================================================
# PHASE 2.9.1 — INVESTIGATION OF MISSING DERIVED RATE FEATURES
# ================================================================
# This cell investigates missing values in:
#   - Flow Byts/s
#   - Flow Pkts/s
#
# It examines the underlying traffic measurements to determine
# whether the missing rates can potentially be reconstructed.
#
# IMPORTANT:
# - No values are modified.
# - No rows are removed.
# - No imputation is performed.
# - This is investigation only.
# ================================================================

import os
import glob
import duckdb
import pandas as pd
import numpy as np

INPUT_DIR = "/content/phase2_schema_normalized"

# If Phase 2.8.2 did not create the directory because we stopped
# before rewriting the files, use the existing cleaned dataset.
if not os.path.exists(INPUT_DIR):
    INPUT_DIR = "/content/phase2_numeric_cleaned"

csv_files = sorted(
    glob.glob(os.path.join(INPUT_DIR, "*.csv"))
)

print("=" * 75)
print("PHASE 2.9.1 — MISSING DERIVED RATE FEATURE INVESTIGATION")
print("=" * 75)

print(f"\nInput directory: {INPUT_DIR}")
print(f"Files found    : {len(csv_files)}")

all_samples = []

# ---------------------------------------------------------------
# Process each file
# ---------------------------------------------------------------

for file_path in csv_files:

    filename = os.path.basename(file_path)

    con = duckdb.connect()

    # -----------------------------------------------------------
    # Select records where either rate is missing or infinite.
    #
    # The Phase 2.5 stage converted infinite values to NULL/NaN,
    # so checking NULL is sufficient here.
    # -----------------------------------------------------------

    query = f"""
        SELECT
            "Timestamp",
            "Label",
            "Flow Duration",
            "Tot Fwd Pkts",
            "Tot Bwd Pkts",
            "TotLen Fwd Pkts",
            "TotLen Bwd Pkts",
            "Flow Byts/s",
            "Flow Pkts/s"
        FROM read_csv_auto(
            '{file_path}',
            header=True,
            ignore_errors=True
        )
        WHERE
            "Flow Byts/s" IS NULL
            OR
            "Flow Pkts/s" IS NULL
        USING SAMPLE 1000
    """

    try:
        sample = con.execute(query).fetchdf()
    except Exception as e:
        print(f"\nWARNING: Could not inspect {filename}")
        print(e)
        con.close()
        continue

    con.close()

    if len(sample) == 0:
        print(f"\n{filename}")
        print("  No missing rate records found.")
        continue

    sample["File"] = filename
    all_samples.append(sample)

    print(f"\n{filename}")
    print(f"  Affected sample records: {len(sample):,}")

# ---------------------------------------------------------------
# Combine samples
# ---------------------------------------------------------------

if not all_samples:

    print("\nNo missing rate records were found.")
    print("Phase 2.9.1 investigation complete.")

else:

    sample_df = pd.concat(
        all_samples,
        ignore_index=True
    )

    # -----------------------------------------------------------
    # Convert underlying features to numeric
    # -----------------------------------------------------------

    numeric_columns = [
        "Flow Duration",
        "Tot Fwd Pkts",
        "Tot Bwd Pkts",
        "TotLen Fwd Pkts",
        "TotLen Bwd Pkts",
        "Flow Byts/s",
        "Flow Pkts/s"
    ]

    for col in numeric_columns:
        sample_df[col] = pd.to_numeric(
            sample_df[col],
            errors="coerce"
        )

    # -----------------------------------------------------------
    # Calculate whether the denominator is zero
    # -----------------------------------------------------------

    sample_df["Flow Duration Zero"] = (
        sample_df["Flow Duration"] == 0
    )

    # -----------------------------------------------------------
    # Calculate theoretical rates where duration > 0.
    #
    # CICFlowMeter Flow Duration is represented in microseconds,
    # therefore the conversion to seconds is:
    #
    # duration_seconds = Flow Duration / 1,000,000
    #
    # These are investigative calculations only.
    # -----------------------------------------------------------

    duration_seconds = (
        sample_df["Flow Duration"] / 1_000_000
    )

    valid_duration = (
        sample_df["Flow Duration"].notna()
        & (sample_df["Flow Duration"] > 0)
    )

    sample_df["Calculated Flow Byts/s"] = np.where(
        valid_duration,
        (
            sample_df["TotLen Fwd Pkts"]
            + sample_df["TotLen Bwd Pkts"]
        ) / duration_seconds,
        np.nan
    )

    sample_df["Calculated Flow Pkts/s"] = np.where(
        valid_duration,
        (
            sample_df["Tot Fwd Pkts"]
            + sample_df["Tot Bwd Pkts"]
        ) / duration_seconds,
        np.nan
    )

    # -----------------------------------------------------------
    # Classify the affected records
    # -----------------------------------------------------------

    sample_df["Potential Reconstruction"] = np.where(
        valid_duration
        & (
            sample_df["Calculated Flow Byts/s"].notna()
            | sample_df["Calculated Flow Pkts/s"].notna()
        ),
        "Possible",
        "Undefined / Requires Imputation"
    )

    # -----------------------------------------------------------
    # Summary
    # -----------------------------------------------------------

    print("\n" + "=" * 75)
    print("INVESTIGATION SUMMARY")
    print("=" * 75)

    print(
        f"Total sampled affected records : "
        f"{len(sample_df):,}"
    )

    print(
        f"Records with Flow Duration > 0 : "
        f"{valid_duration.sum():,}"
    )

    print(
        f"Records with Flow Duration = 0 : "
        f"{sample_df['Flow Duration Zero'].sum():,}"
    )

    print("\nPotential treatment:")
    display(
        sample_df[
            "Potential Reconstruction"
        ].value_counts()
        .rename_axis("Category")
        .reset_index(name="Count")
    )

    # -----------------------------------------------------------
    # Display representative records
    # -----------------------------------------------------------

    print("\n" + "=" * 75)
    print("REPRESENTATIVE AFFECTED RECORDS")
    print("=" * 75)

    display(
        sample_df[
            [
                "File",
                "Timestamp",
                "Label",
                "Flow Duration",
                "Tot Fwd Pkts",
                "Tot Bwd Pkts",
                "TotLen Fwd Pkts",
                "TotLen Bwd Pkts",
                "Flow Byts/s",
                "Flow Pkts/s",
                "Calculated Flow Byts/s",
                "Calculated Flow Pkts/s",
                "Potential Reconstruction"
            ]
        ].head(50)
    )

print("\n" + "=" * 75)
print("PHASE 2.9.1 STATUS")
print("=" * 75)

print("Investigation completed.")
print("No dataset values were modified.")
print("No rows were removed.")
print("No imputation was performed.")

PHASE 2.9.1 — MISSING DERIVED RATE FEATURE INVESTIGATION

Input directory: /content/phase2_schema_normalized
Files found    : 3


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))


Friday-02-03-2018_TrafficForML_CICFlowMeter.csv
  Affected sample records: 10


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))


Friday-16-02-2018_TrafficForML_CICFlowMeter.csv
  No missing rate records found.


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))


Friday-23-02-2018_TrafficForML_CICFlowMeter.csv
  Affected sample records: 9

INVESTIGATION SUMMARY
Total sampled affected records : 19
Records with Flow Duration > 0 : 0
Records with Flow Duration = 0 : 19

Potential treatment:


,Category,Count
0,Undefined / Requires Imputation,19



REPRESENTATIVE AFFECTED RECORDS


,File,Timestamp,Label,Flow Duration,Tot Fwd Pkts,Tot Bwd Pkts,TotLen Fwd Pkts,TotLen Bwd Pkts,Flow Byts/s,Flow Pkts/s,Calculated Flow Byts/s,Calculated Flow Pkts/s,Potential Reconstruction
0,Friday-02-03-2018_TrafficForML_CICFlowMeter.csv,2018-03-02 02:31:38,Benign,0,2,0,0,0.0,NaN,NaN,NaN,NaN,Undefined / Requires Imputation
1,Friday-02-03-2018_TrafficForML_CICFlowMeter.csv,2018-03-02 02:53:05,Benign,0,2,0,0,0.0,NaN,NaN,NaN,NaN,Undefined / Requires Imputation
2,Friday-02-03-2018_TrafficForML_CICFlowMeter.csv,2018-03-02 11:06:42,Benign,0,2,0,53,0.0,NaN,NaN,NaN,NaN,Undefined / Requires Imputation
3,Friday-02-03-2018_TrafficForML_CICFlowMeter.csv,2018-03-02 11:08:13,Benign,0,2,0,0,0.0,NaN,NaN,NaN,NaN,Undefined / Requires Imputation
4,Friday-02-03-2018_TrafficForML_CICFlowMeter.csv,2018-03-02 01:11:48,Benign,0,2,0,0,0.0,NaN,NaN,NaN,NaN,Undefined / Requires Imputation
5,Friday-02-03-2018_TrafficForML_CICFlowMeter.csv,2018-03-02 09:30:37,Benign,0,2,0,0,0.0,NaN,NaN,NaN,NaN,Undefined / Requires Imputation
6,Friday-02-03-2018_TrafficForML_CICFlowMeter.csv,2018-03-02 03:42:38,Benign,0,2,0,31,0.0,NaN,NaN,NaN,NaN,Undefined / Requires Imputation
7,Friday-02-03-2018_TrafficForML_CICFlowMeter.csv,2018-03-02 04:30:26,Benign,0,2,0,31,0.0,NaN,NaN,NaN,NaN,Undefined / Requires Imputation
8,Friday-02-03-2018_TrafficForML_CICFlowMeter.csv,2018-03-02 08:54:15,Benign,0,2,0,0,0.0,NaN,NaN,NaN,NaN,Undefined / Requires Imputation
9,Friday-02-03-2018_TrafficForML_CICFlowMeter.csv,2018-03-02 08:56:25,Benign,0,2,0,31,0.0,NaN,NaN,NaN,NaN,Undefined / Requires Imputation



PHASE 2.9.1 STATUS
Investigation completed.
No dataset values were modified.
No rows were removed.
No imputation was performed.



# PHASE 2.10.1 — FINAL PREPROCESSING VALIDATION

# Final validation of the Phase 2 cleaned CIC-IDS2018 dataset.

 This cell:
   1. Uses the final Phase 2 output.
   2. Validates row counts.
   3. Checks duplicate records.
   4. Checks NaN and infinite values.
  5. Checks invalid negative IAT values.
   6. Checks required schema.
  7. Verifies chronological ordering.
   8. Summarizes labels.
   9. Creates a final Phase 2 metadata JSON file.

# IMPORTANT:
- No dataset values are modified.
- No rows are removed.
 - Flow Byts/s and Flow Pkts/s NaNs are intentionally retained.


In [7]:
# ================================================================
# PHASE 2.10.1 — FINAL PREPROCESSING VALIDATION
# ================================================================
# Final validation of the Phase 2 cleaned CIC-IDS2018 dataset.
#
# This cell:
#   1. Validates all 10 processed files.
#   2. Checks final row counts.
#   3. Checks required schema.
#   4. Checks remaining NaN values in known rate features.
#   5. Checks remaining infinite values.
#   6. Checks invalid negative IAT values.
#   7. Checks timestamp validity.
#   8. Checks chronological ordering.
#   9. Checks label integrity.
#  10. Saves final Phase 2 metadata.
#
# IMPORTANT:
# - No data is modified.
# - No rows are removed.
# - No imputation is performed.
# - Duplicate removal is NOT repeated because it was completed
#   in Phase 2.6.
# ================================================================

import os
import glob
import json
from datetime import datetime

import duckdb
import pandas as pd

# ---------------------------------------------------------------
# Configuration
# ---------------------------------------------------------------

INPUT_DIR = "/content/phase2_numeric_cleaned"
FINAL_METADATA_FILE = "/content/phase2_final_metadata.json"

csv_files = sorted(
    glob.glob(os.path.join(INPUT_DIR, "*.csv"))
)

print("=" * 75)
print("PHASE 2.10.1 — FINAL PREPROCESSING VALIDATION")
print("=" * 75)

print(f"\nInput directory : {INPUT_DIR}")
print(f"Files found     : {len(csv_files)}")

# ---------------------------------------------------------------
# Required columns
# ---------------------------------------------------------------

required_columns = {
    "Timestamp",
    "Label",
    "Flow Duration",
    "Flow IAT Min",
    "Fwd IAT Min",
    "Flow Byts/s",
    "Flow Pkts/s"
}

# ---------------------------------------------------------------
# Results
# ---------------------------------------------------------------

results = []

grand_total_rows = 0
grand_missing_rates = 0
grand_infinite_rates = 0
grand_invalid_iat = 0
grand_invalid_timestamps = 0
grand_empty_labels = 0

overall_schema_ok = True
overall_chronology_ok = True
overall_timestamp_ok = True
overall_label_ok = True

# ================================================================
# FILE-BY-FILE VALIDATION
# ================================================================

for index, file_path in enumerate(csv_files, start=1):

    filename = os.path.basename(file_path)

    print("\n" + "-" * 75)
    print(f"[{index}/{len(csv_files)}] {filename}")
    print("-" * 75)

    con = duckdb.connect()

    # -----------------------------------------------------------
    # 1. Schema validation
    # -----------------------------------------------------------

    schema_df = con.execute(f"""
        DESCRIBE
        SELECT *
        FROM read_csv_auto(
            '{file_path}',
            header=True
        )
    """).fetchdf()

    columns = schema_df["column_name"].tolist()

    missing_required = [
        col for col in required_columns
        if col not in columns
    ]

    schema_ok = len(missing_required) == 0

    if not schema_ok:
        overall_schema_ok = False

    # -----------------------------------------------------------
    # 2. Row count
    # -----------------------------------------------------------

    row_count = con.execute(f"""
        SELECT COUNT(*)
        FROM read_csv_auto(
            '{file_path}',
            header=True,
            ignore_errors=True
        )
    """).fetchone()[0]

    # -----------------------------------------------------------
    # 3. Missing values in derived rate features
    # -----------------------------------------------------------

    missing_result = con.execute(f"""
        SELECT
            SUM(
                CASE
                    WHEN "Flow Byts/s" IS NULL
                    THEN 1
                    ELSE 0
                END
            ),

            SUM(
                CASE
                    WHEN "Flow Pkts/s" IS NULL
                    THEN 1
                    ELSE 0
                END
            )
        FROM read_csv_auto(
            '{file_path}',
            header=True,
            ignore_errors=True
        )
    """).fetchone()

    flow_bytes_missing = int(missing_result[0] or 0)
    flow_packets_missing = int(missing_result[1] or 0)

    missing_rates = (
        flow_bytes_missing +
        flow_packets_missing
    )

    # -----------------------------------------------------------
    # 4. Infinite values
    # -----------------------------------------------------------

    infinite_result = con.execute(f"""
        SELECT
            SUM(
                CASE
                    WHEN isinf(
                        TRY_CAST("Flow Byts/s" AS DOUBLE)
                    )
                    THEN 1
                    ELSE 0
                END
            ),

            SUM(
                CASE
                    WHEN isinf(
                        TRY_CAST("Flow Pkts/s" AS DOUBLE)
                    )
                    THEN 1
                    ELSE 0
                END
            )
        FROM read_csv_auto(
            '{file_path}',
            header=True,
            ignore_errors=True
        )
    """).fetchone()

    infinite_rates = (
        int(infinite_result[0] or 0)
        +
        int(infinite_result[1] or 0)
    )

    # -----------------------------------------------------------
    # 5. Invalid negative IAT values
    # -----------------------------------------------------------

    invalid_iat = con.execute(f"""
        SELECT COUNT(*)
        FROM read_csv_auto(
            '{file_path}',
            header=True,
            ignore_errors=True
        )
        WHERE
            TRY_CAST("Flow IAT Min" AS DOUBLE) < 0
            OR
            TRY_CAST("Fwd IAT Min" AS DOUBLE) < 0
    """).fetchone()[0]

    # -----------------------------------------------------------
    # 6. Invalid timestamps
    # -----------------------------------------------------------

    invalid_timestamps = con.execute(f"""
        SELECT COUNT(*)
        FROM read_csv_auto(
            '{file_path}',
            header=True,
            ignore_errors=True
        )
        WHERE
            TRY_CAST("Timestamp" AS TIMESTAMP) IS NULL
    """).fetchone()[0]

    # -----------------------------------------------------------
    # 7. Empty labels
    # -----------------------------------------------------------

    empty_labels = con.execute(f"""
        SELECT COUNT(*)
        FROM read_csv_auto(
            '{file_path}',
            header=True,
            ignore_errors=True
        )
        WHERE
            "Label" IS NULL
            OR
            TRIM(CAST("Label" AS VARCHAR)) = ''
    """).fetchone()[0]

    # -----------------------------------------------------------
    # 8. Chronological ordering
    #
    # IMPORTANT:
    # We generate a sequence number from the CSV input order.
    # We then compare each timestamp with the previous timestamp.
    #
    # No rowid is assumed.
    # -----------------------------------------------------------

    chronology_result = con.execute(f"""
        SELECT COUNT(*)
        FROM (
            SELECT
                TRY_CAST("Timestamp" AS TIMESTAMP) AS ts,

                LAG(
                    TRY_CAST("Timestamp" AS TIMESTAMP)
                ) OVER (
                    ORDER BY input_sequence
                ) AS previous_ts

            FROM (
                SELECT
                    *,
                    ROW_NUMBER() OVER () AS input_sequence
                FROM read_csv_auto(
                    '{file_path}',
                    header=True,
                    ignore_errors=True
                )
            )
        )
        WHERE
            previous_ts IS NOT NULL
            AND ts < previous_ts
    """).fetchone()[0]

    chronology_ok = chronology_result == 0

    # -----------------------------------------------------------
    # Update overall flags
    # -----------------------------------------------------------

    if not chronology_ok:
        overall_chronology_ok = False

    if invalid_timestamps > 0:
        overall_timestamp_ok = False

    if empty_labels > 0:
        overall_label_ok = False

    # -----------------------------------------------------------
    # Update totals
    # -----------------------------------------------------------

    grand_total_rows += row_count
    grand_missing_rates += missing_rates
    grand_infinite_rates += infinite_rates
    grand_invalid_iat += invalid_iat
    grand_invalid_timestamps += invalid_timestamps
    grand_empty_labels += empty_labels

    # -----------------------------------------------------------
    # Store result
    # -----------------------------------------------------------

    results.append({
        "File": filename,
        "Rows": row_count,
        "Schema OK": schema_ok,
        "Flow Byts/s NaN": flow_bytes_missing,
        "Flow Pkts/s NaN": flow_packets_missing,
        "Infinite Rate Values": infinite_rates,
        "Invalid IAT Negatives": invalid_iat,
        "Invalid Timestamps": invalid_timestamps,
        "Empty Labels": empty_labels,
        "Chronological": chronology_ok
    })

    # -----------------------------------------------------------
    # Display result
    # -----------------------------------------------------------

    print(f"Rows                  : {row_count:,}")
    print(f"Schema OK             : {schema_ok}")
    print(f"Flow Byts/s NaN       : {flow_bytes_missing:,}")
    print(f"Flow Pkts/s NaN       : {flow_packets_missing:,}")
    print(f"Infinite rate values  : {infinite_rates:,}")
    print(f"Invalid IAT negatives : {invalid_iat:,}")
    print(f"Invalid timestamps    : {invalid_timestamps:,}")
    print(f"Empty labels          : {empty_labels:,}")
    print(f"Chronological         : {chronology_ok}")

    con.close()

# ================================================================
# FINAL VALIDATION TABLE
# ================================================================

results_df = pd.DataFrame(results)

print("\n" + "=" * 75)
print("FINAL VALIDATION TABLE")
print("=" * 75)

display(results_df)

# ================================================================
# OVERALL SUMMARY
# ================================================================

print("\n" + "=" * 75)
print("OVERALL PHASE 2 VALIDATION")
print("=" * 75)

print(f"Files validated              : {len(csv_files)}")
print(f"Total final rows             : {grand_total_rows:,}")
print(f"Missing rate cells           : {grand_missing_rates:,}")
print(f"Infinite rate values         : {grand_infinite_rates:,}")
print(f"Invalid IAT negatives        : {grand_invalid_iat:,}")
print(f"Invalid timestamps            : {grand_invalid_timestamps:,}")
print(f"Empty labels                 : {grand_empty_labels:,}")

print(f"\nSchema validation             : {overall_schema_ok}")
print(f"Timestamp validation          : {overall_timestamp_ok}")
print(f"Chronological validation     : {overall_chronology_ok}")
print(f"Label validation              : {overall_label_ok}")

# ================================================================
# FINAL PHASE 2 STATUS
# ================================================================

phase2_pass = (
    len(csv_files) == 10
    and grand_infinite_rates == 0
    and grand_invalid_iat == 0
    and grand_invalid_timestamps == 0
    and grand_empty_labels == 0
    and overall_schema_ok
    and overall_timestamp_ok
    and overall_chronology_ok
    and overall_label_ok
)

print("\n" + "=" * 75)
print("PHASE 2 FINAL STATUS")
print("=" * 75)

if phase2_pass:

    print("STATUS: PASS")
    print("\nPHASE 2 — DATA PREPROCESSING: COMPLETE")

else:

    print("STATUS: REVIEW REQUIRED")

# ================================================================
# FINAL METADATA
# ================================================================

final_metadata = {
    "project": (
        "SIH26153 — AI based Network Attack "
        "Forecasting from Network Traffic Data"
    ),

    "dataset": "CIC-IDS2018",

    "phase": "Phase 2 — Data Preprocessing",

    "validation_date": datetime.now().isoformat(),

    "input_directory": INPUT_DIR,

    "files_validated": len(csv_files),

    "final_total_rows": int(grand_total_rows),

    "preprocessing_operations_completed": [
        "Dataset metadata and provenance recorded",
        "Timestamp conversion performed",
        "Malformed/header-like rows removed",
        "Suspicious 1970 timestamp records removed",
        "Chronological ordering established",
        "Infinite rate values converted to missing values",
        "Exact duplicate records removed",
        "Invalid negative IAT record removed",
        "Schema validation completed",
        "Numeric feature validation completed",
        "Missing derived rate values retained for later treatment"
    ],

    "final_validation": {
        "schema_valid": bool(overall_schema_ok),
        "timestamp_valid": bool(overall_timestamp_ok),
        "chronological": bool(overall_chronology_ok),
        "labels_valid": bool(overall_label_ok),
        "infinite_values_remaining": int(grand_infinite_rates),
        "invalid_iat_negatives_remaining": int(grand_invalid_iat),
        "invalid_timestamps_remaining": int(grand_invalid_timestamps),
        "empty_labels_remaining": int(grand_empty_labels),
        "missing_rate_cells": int(grand_missing_rates)
    },

    "missing_value_policy": {
        "features": [
            "Flow Byts/s",
            "Flow Pkts/s"
        ],
        "policy": (
            "Retain undefined values as missing during "
            "dataset preprocessing"
        ),
        "future_treatment": (
            "Training-only imputation within ML "
            "preprocessing pipeline"
        )
    },

    "duplicate_policy": {
        "status": (
            "Exact duplicates removed during Phase 2.6"
        ),
        "final_validation": (
            "Not rescanned during final validation"
        )
    },

    "storage_policy": {
        "raw_dataset": "Private Hugging Face repository",
        "phase_2_intermediate": "Colab temporary storage",
        "processed_dataset": (
            "Private Hugging Face repository after Phase 3"
        )
    },

    "phase_2_status": (
        "COMPLETE"
        if phase2_pass
        else
        "REVIEW REQUIRED"
    )
}

with open(
    FINAL_METADATA_FILE,
    "w",
    encoding="utf-8"
) as f:

    json.dump(
        final_metadata,
        f,
        indent=4
    )

print("\nFinal metadata saved to:")
print(FINAL_METADATA_FILE)

PHASE 2.10.1 — FINAL PREPROCESSING VALIDATION

Input directory : /content/phase2_numeric_cleaned
Files found     : 10

---------------------------------------------------------------------------
[1/10] Friday-02-03-2018_TrafficForML_CICFlowMeter.csv
---------------------------------------------------------------------------
Rows                  : 1,043,116
Schema OK             : True
Flow Byts/s NaN       : 4,044
Flow Pkts/s NaN       : 4,044
Infinite rate values  : 0
Invalid IAT negatives : 0
Invalid timestamps    : 0
Empty labels          : 0
Chronological         : False

---------------------------------------------------------------------------
[2/10] Friday-16-02-2018_TrafficForML_CICFlowMeter.csv
---------------------------------------------------------------------------


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Rows                  : 900,988
Schema OK             : True
Flow Byts/s NaN       : 0
Flow Pkts/s NaN       : 0
Infinite rate values  : 0
Invalid IAT negatives : 0
Invalid timestamps    : 0
Empty labels          : 0
Chronological         : False

---------------------------------------------------------------------------
[3/10] Friday-23-02-2018_TrafficForML_CICFlowMeter.csv
---------------------------------------------------------------------------
Rows                  : 1,045,960
Schema OK             : True
Flow Byts/s NaN       : 5,702
Flow Pkts/s NaN       : 5,702
Infinite rate values  : 0
Invalid IAT negatives : 0
Invalid timestamps    : 0
Empty labels          : 0
Chronological         : False

---------------------------------------------------------------------------
[4/10] Thuesday-20-02-2018_TrafficForML_CICFlowMeter.csv
---------------------------------------------------------------------------


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

... (12 outputs left out)

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Rows                  : 1,031,018
Schema OK             : True
Flow Byts/s NaN       : 0
Flow Pkts/s NaN       : 0
Infinite rate values  : 0
Invalid IAT negatives : 0
Invalid timestamps    : 0
Empty labels          : 0
Chronological         : False

---------------------------------------------------------------------------
[10/10] Wednesday-28-02-2018_TrafficForML_CICFlowMeter.csv
---------------------------------------------------------------------------
Rows                  : 606,982
Schema OK             : True
Flow Byts/s NaN       : 4,786
Flow Pkts/s NaN       : 4,786
Infinite rate values  : 0
Invalid IAT negatives : 0
Invalid timestamps    : 0
Empty labels          : 0
Chronological         : False

FINAL VALIDATION TABLE


,File,Rows,Schema OK,Flow Byts/s NaN,Flow Pkts/s NaN,Infinite Rate Values,Invalid IAT Negatives,Invalid Timestamps,Empty Labels,Chronological
0,Friday-02-03-2018_TrafficForML_CICFlowMeter.csv,1043116,True,4044,4044,0,0,0,0,False
1,Friday-16-02-2018_TrafficForML_CICFlowMeter.csv,900988,True,0,0,0,0,0,0,False
2,Friday-23-02-2018_TrafficForML_CICFlowMeter.csv,1045960,True,5702,5702,0,0,0,0,False
3,Thuesday-20-02-2018_TrafficForML_CICFlowMeter.csv,7948746,True,59452,59452,0,0,0,0,False
4,Thursday-01-03-2018_TrafficForML_CICFlowMeter.csv,331027,True,2917,2917,0,0,0,0,False
5,Thursday-15-02-2018_TrafficForML_CICFlowMeter.csv,1046154,True,8022,8022,0,0,0,0,False
6,Thursday-22-02-2018_TrafficForML_CICFlowMeter.csv,1045288,True,5604,5604,0,0,0,0,False
7,Wednesday-14-02-2018_TrafficForML_CICFlowMeter...,822942,True,3821,3821,0,0,0,0,False
8,Wednesday-21-02-2018_TrafficForML_CICFlowMeter...,1031018,True,0,0,0,0,0,0,False
9,Wednesday-28-02-2018_TrafficForML_CICFlowMeter...,606982,True,4786,4786,0,0,0,0,False



OVERALL PHASE 2 VALIDATION
Files validated              : 10
Total final rows             : 15,822,221
Missing rate cells           : 188,696
Infinite rate values         : 0
Invalid IAT negatives        : 0
Invalid timestamps            : 0
Empty labels                 : 0

Schema validation             : True
Timestamp validation          : True
Chronological validation     : False
Label validation              : True

PHASE 2 FINAL STATUS
STATUS: REVIEW REQUIRED

Final metadata saved to:
/content/phase2_final_metadata.json


#Phase 2.10.2 — Reliable Chronological Verification

We'll do a targeted verification instead of another expensive full chronological comparison.

For each file, we'll inspect:

first few timestamps

last few timestamps

minimum timestamp

maximum timestamp

whether timestamps are within the expected capture period

This will tell us whether the Phase 2.4 chronological files were actually preserved.

In [8]:
# ================================================================
# PHASE 2.10.2 — RELIABLE CHRONOLOGICAL VERIFICATION
# ================================================================
# This cell verifies the chronological state of the Phase 2 files
# without performing another complete dataset sort.
#
# It checks:
#   1. First timestamps in each file
#   2. Last timestamps in each file
#   3. Minimum timestamp
#   4. Maximum timestamp
#   5. Whether the first/last timestamps are sensible
#
# IMPORTANT:
# - No data is modified.
# - No files are rewritten.
# - No sorting is performed.
# ================================================================

import os
import glob
import duckdb
import pandas as pd

INPUT_DIR = "/content/phase2_numeric_cleaned"

csv_files = sorted(
    glob.glob(os.path.join(INPUT_DIR, "*.csv"))
)

print("=" * 75)
print("PHASE 2.10.2 — RELIABLE CHRONOLOGICAL VERIFICATION")
print("=" * 75)

verification_results = []

for index, file_path in enumerate(csv_files, start=1):

    filename = os.path.basename(file_path)

    print("\n" + "-" * 75)
    print(f"[{index}/{len(csv_files)}] {filename}")
    print("-" * 75)

    con = duckdb.connect()

    # -----------------------------------------------------------
    # First timestamp
    # -----------------------------------------------------------

    first_rows = con.execute(f"""
        SELECT
            "Timestamp"
        FROM read_csv_auto(
            '{file_path}',
            header=True,
            ignore_errors=True
        )
        LIMIT 5
    """).fetchdf()

    # -----------------------------------------------------------
    # Last timestamp
    #
    # Read the last 5 records using LIMIT/OFFSET based on row count.
    # -----------------------------------------------------------

    row_count = con.execute(f"""
        SELECT COUNT(*)
        FROM read_csv_auto(
            '{file_path}',
            header=True,
            ignore_errors=True
        )
    """).fetchone()[0]

    last_rows = con.execute(f"""
        SELECT
            "Timestamp"
        FROM read_csv_auto(
            '{file_path}',
            header=True,
            ignore_errors=True
        )
        LIMIT 5 OFFSET {max(0, row_count - 5)}
    """).fetchdf()

    # -----------------------------------------------------------
    # Global timestamp range
    # -----------------------------------------------------------

    timestamp_range = con.execute(f"""
        SELECT
            MIN(
                TRY_CAST("Timestamp" AS TIMESTAMP)
            ) AS min_timestamp,

            MAX(
                TRY_CAST("Timestamp" AS TIMESTAMP)
            ) AS max_timestamp
        FROM read_csv_auto(
            '{file_path}',
            header=True,
            ignore_errors=True
        )
    """).fetchone()

    min_timestamp = timestamp_range[0]
    max_timestamp = timestamp_range[1]

    # -----------------------------------------------------------
    # Convert first/last timestamps
    # -----------------------------------------------------------

    first_timestamp_values = pd.to_datetime(
        first_rows["Timestamp"],
        errors="coerce"
    )

    last_timestamp_values = pd.to_datetime(
        last_rows["Timestamp"],
        errors="coerce"
    )

    first_is_ordered = (
        first_timestamp_values.is_monotonic_increasing
    )

    last_is_ordered = (
        last_timestamp_values.is_monotonic_increasing
    )

    # -----------------------------------------------------------
    # Print results
    # -----------------------------------------------------------

    print("\nFirst 5 timestamps:")
    print(first_rows.to_string(index=False))

    print("\nLast 5 timestamps:")
    print(last_rows.to_string(index=False))

    print("\nTimestamp range:")
    print(f"  Minimum: {min_timestamp}")
    print(f"  Maximum: {max_timestamp}")

    print("\nLocal boundary checks:")
    print(f"  First 5 ordered : {first_is_ordered}")
    print(f"  Last 5 ordered  : {last_is_ordered}")

    verification_results.append({
        "File": filename,
        "Rows": row_count,
        "Minimum Timestamp": str(min_timestamp),
        "Maximum Timestamp": str(max_timestamp),
        "First 5 Ordered": first_is_ordered,
        "Last 5 Ordered": last_is_ordered
    })

    con.close()

# ================================================================
# FINAL TABLE
# ================================================================

verification_df = pd.DataFrame(
    verification_results
)

print("\n" + "=" * 75)
print("CHRONOLOGICAL VERIFICATION SUMMARY")
print("=" * 75)

display(verification_df)

print("\n" + "=" * 75)
print("PHASE 2.10.2 STATUS")
print("=" * 75)

print(
    "Boundary verification completed."
)

print(
    "\nNo files were modified."
)

PHASE 2.10.2 — RELIABLE CHRONOLOGICAL VERIFICATION

---------------------------------------------------------------------------
[1/10] Friday-02-03-2018_TrafficForML_CICFlowMeter.csv
---------------------------------------------------------------------------

First 5 timestamps:
          Timestamp
2018-03-02 01:00:00
2018-03-02 01:00:00
2018-03-02 01:00:00
2018-03-02 01:00:00
2018-03-02 01:00:00

Last 5 timestamps:
          Timestamp
2018-03-02 12:51:59
2018-03-02 12:51:59
2018-03-02 12:51:59
2018-03-02 12:51:59
2018-03-02 12:49:58

Timestamp range:
  Minimum: 2018-03-02 01:00:00
  Maximum: 2018-03-02 12:59:59

Local boundary checks:
  First 5 ordered : True
  Last 5 ordered  : False

---------------------------------------------------------------------------
[2/10] Friday-16-02-2018_TrafficForML_CICFlowMeter.csv
---------------------------------------------------------------------------

First 5 timestamps:
          Timestamp
2018-02-16 01:45:30
2018-02-16 01:45:30
2018-02-16 01:45

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))


First 5 timestamps:
          Timestamp
2018-02-23 01:11:57
2018-02-23 01:11:58
2018-02-23 01:11:58
2018-02-23 01:11:58
2018-02-23 01:11:58

Last 5 timestamps:
          Timestamp
2018-02-23 12:55:15
2018-02-23 12:55:16
2018-02-23 12:55:16
2018-02-23 12:55:16
2018-02-23 12:55:16

Timestamp range:
  Minimum: 2018-02-23 01:00:00
  Maximum: 2018-02-23 12:59:59

Local boundary checks:
  First 5 ordered : True
  Last 5 ordered  : True

---------------------------------------------------------------------------
[4/10] Thuesday-20-02-2018_TrafficForML_CICFlowMeter.csv
---------------------------------------------------------------------------


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

... (12 outputs left out)

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))


First 5 timestamps:
          Timestamp
2018-02-28 01:00:01
2018-02-28 01:00:02
2018-02-28 01:00:02
2018-02-28 01:00:03
2018-02-28 01:00:03

Last 5 timestamps:
          Timestamp
2018-02-28 12:48:51
2018-02-28 12:48:51
2018-02-28 12:48:51
2018-02-28 12:48:51
2018-02-28 12:48:51

Timestamp range:
  Minimum: 2018-02-28 01:00:00
  Maximum: 2018-02-28 12:59:59

Local boundary checks:
  First 5 ordered : True
  Last 5 ordered  : True

CHRONOLOGICAL VERIFICATION SUMMARY


,File,Rows,Minimum Timestamp,Maximum Timestamp,First 5 Ordered,Last 5 Ordered
0,Friday-02-03-2018_TrafficForML_CICFlowMeter.csv,1043116,2018-03-02 01:00:00,2018-03-02 12:59:59,True,False
1,Friday-16-02-2018_TrafficForML_CICFlowMeter.csv,900988,2018-02-16 01:00:32,2018-02-16 12:58:24,True,True
2,Friday-23-02-2018_TrafficForML_CICFlowMeter.csv,1045960,2018-02-23 01:00:00,2018-02-23 12:59:59,True,True
3,Thuesday-20-02-2018_TrafficForML_CICFlowMeter.csv,7948746,2018-02-20 01:00:00,2018-02-20 12:59:59,True,True
4,Thursday-01-03-2018_TrafficForML_CICFlowMeter.csv,331027,2018-03-01 01:00:00,2018-03-01 12:59:59,True,True
5,Thursday-15-02-2018_TrafficForML_CICFlowMeter.csv,1046154,2018-02-15 01:00:00,2018-02-15 12:59:59,True,True
6,Thursday-22-02-2018_TrafficForML_CICFlowMeter.csv,1045288,2018-02-22 01:00:00,2018-02-22 12:59:59,True,False
7,Wednesday-14-02-2018_TrafficForML_CICFlowMeter...,822942,2018-02-14 01:00:00,2018-02-14 12:59:59,True,True
8,Wednesday-21-02-2018_TrafficForML_CICFlowMeter...,1031018,2018-02-21 01:55:46,2018-02-21 10:43:21,True,True
9,Wednesday-28-02-2018_TrafficForML_CICFlowMeter...,606982,2018-02-28 01:00:00,2018-02-28 12:59:59,True,True



PHASE 2.10.2 STATUS
Boundary verification completed.

No files were modified.


#Phase 2.10.3 — Final Chronological Re-Sort & Verification

Yes. This will be the final ordering operation of Phase 2.

We will take the fully cleaned dataset from:

/content/phase2_numeric_cleaned

and create:

/content/phase2_final

The code will:

Sort every file by Timestamp.
Preserve all columns and cleaned values.
Write the final files to a new directory.
Verify the timestamp ordering using the final files.
Report row counts and timestamp ranges.
Not modify the source directory.

In [9]:
# ============================================================================
# PHASE 2.10.3 — FINAL CHRONOLOGICAL RE-SORT & VERIFICATION
# ============================================================================

import os
import glob
import duckdb
import pandas as pd

# ---------------------------------------------------------------------------
# 1. Define input and output directories
# ---------------------------------------------------------------------------

INPUT_DIR = "/content/phase2_numeric_cleaned"
OUTPUT_DIR = "/content/phase2_final"

os.makedirs(OUTPUT_DIR, exist_ok=True)

csv_files = sorted(glob.glob(os.path.join(INPUT_DIR, "*.csv")))

print("=" * 80)
print("PHASE 2.10.3 — FINAL CHRONOLOGICAL RE-SORT & VERIFICATION")
print("=" * 80)

print(f"\nInput directory : {INPUT_DIR}")
print(f"Output directory: {OUTPUT_DIR}")
print(f"Files found     : {len(csv_files)}")

# ---------------------------------------------------------------------------
# 2. Re-sort every cleaned file by Timestamp
#
# DuckDB is used because the dataset is very large and Pandas would require
# excessive RAM for several of these files.
# ---------------------------------------------------------------------------

con = duckdb.connect()

results = []

for i, input_file in enumerate(csv_files, start=1):

    file_name = os.path.basename(input_file)
    output_file = os.path.join(OUTPUT_DIR, file_name)

    print("\n" + "-" * 80)
    print(f"[{i}/{len(csv_files)}] {file_name}")
    print("-" * 80)

    # Read CSV and sort by the already-cleaned Timestamp column.
    #
    # ORDER BY Timestamp is the authoritative chronological ordering.
    # NULL timestamps are placed last, although previous Phase 2 validation
    # established that invalid timestamps had already been removed.
    #
    # The filename itself is escaped to safely handle special characters.
    escaped_input = input_file.replace("'", "''")
    escaped_output = output_file.replace("'", "''")

    query = f"""
        COPY (
            SELECT *
            FROM read_csv_auto(
                '{escaped_input}',
                header = true,
                ignore_errors = false
            )
            ORDER BY TRY_CAST("Timestamp" AS TIMESTAMP)
        )
        TO '{escaped_output}'
        WITH (
            HEADER,
            DELIMITER ','
        );
    """

    con.execute(query)

    # -----------------------------------------------------------------------
    # 3. Basic final-file statistics
    # -----------------------------------------------------------------------

    escaped_final = output_file.replace("'", "''")

    stats_query = f"""
        SELECT
            COUNT(*) AS row_count,
            MIN(TRY_CAST("Timestamp" AS TIMESTAMP)) AS min_timestamp,
            MAX(TRY_CAST("Timestamp" AS TIMESTAMP)) AS max_timestamp,
            COUNT(*) FILTER (
                WHERE TRY_CAST("Timestamp" AS TIMESTAMP) IS NULL
            ) AS invalid_timestamp_count
        FROM read_csv_auto(
            '{escaped_final}',
            header = true,
            ignore_errors = false
        );
    """

    stats = con.execute(stats_query).fetchone()

    row_count = int(stats[0])
    min_timestamp = stats[1]
    max_timestamp = stats[2]
    invalid_timestamp_count = int(stats[3])

    # -----------------------------------------------------------------------
    # 4. Verify chronological ordering efficiently
    #
    # We compare the maximum timestamp in each sequential chunk with the
    # minimum timestamp in the next chunk.
    #
    # If every chunk boundary is correctly ordered, the complete file is
    # chronologically ordered.
    #
    # A sequential row number is generated with parallel CSV reading disabled
    # so that the physical CSV order is respected during verification.
    # -----------------------------------------------------------------------

    chunk_size = 250_000

    order_query = f"""
        WITH ordered_rows AS (
            SELECT
                TRY_CAST("Timestamp" AS TIMESTAMP) AS ts,
                ROW_NUMBER() OVER () AS row_num
            FROM read_csv_auto(
                '{escaped_final}',
                header = true,
                ignore_errors = false,
                parallel = false
            )
        ),
        chunked AS (
            SELECT
                FLOOR((row_num - 1) / {chunk_size}) AS chunk_id,
                MIN(ts) AS chunk_min,
                MAX(ts) AS chunk_max
            FROM ordered_rows
            GROUP BY FLOOR((row_num - 1) / {chunk_size})
        ),
        boundaries AS (
            SELECT
                chunk_id,
                chunk_max,
                LEAD(chunk_min) OVER (ORDER BY chunk_id) AS next_chunk_min
            FROM chunked
        )
        SELECT COUNT(*)
        FROM boundaries
        WHERE next_chunk_min IS NOT NULL
          AND chunk_max > next_chunk_min;
    """

    ordering_violations = int(con.execute(order_query).fetchone()[0])

    chronological = ordering_violations == 0

    results.append({
        "File": file_name,
        "Rows": row_count,
        "Minimum Timestamp": str(min_timestamp),
        "Maximum Timestamp": str(max_timestamp),
        "Invalid Timestamps": invalid_timestamp_count,
        "Ordering Violations": ordering_violations,
        "Chronological": chronological
    })

    print(f"Rows                 : {row_count:,}")
    print(f"Minimum Timestamp    : {min_timestamp}")
    print(f"Maximum Timestamp    : {max_timestamp}")
    print(f"Invalid Timestamps   : {invalid_timestamp_count:,}")
    print(f"Ordering Violations  : {ordering_violations:,}")
    print(f"Chronological        : {chronological}")

# ---------------------------------------------------------------------------
# 5. Final summary
# ---------------------------------------------------------------------------

results_df = pd.DataFrame(results)

print("\n" + "=" * 80)
print("FINAL CHRONOLOGICAL SORT SUMMARY")
print("=" * 80)

print(results_df.to_string(index=False))

print("\n" + "=" * 80)

all_chronological = results_df["Chronological"].all()
total_rows = results_df["Rows"].sum()
total_invalid_timestamps = results_df["Invalid Timestamps"].sum()
total_ordering_violations = results_df["Ordering Violations"].sum()

print(f"Files processed       : {len(results_df)}")
print(f"Total final rows      : {total_rows:,}")
print(f"Invalid timestamps    : {total_invalid_timestamps:,}")
print(f"Ordering violations   : {total_ordering_violations:,}")
print(f"All files chronological: {all_chronological}")

if all_chronological and total_invalid_timestamps == 0:
    print("\nSTATUS: PHASE 2 CHRONOLOGICAL ORDERING VERIFIED")
    print("Final dataset directory:", OUTPUT_DIR)
else:
    print("\nSTATUS: REVIEW REQUIRED")
    print("Do not proceed to Phase 3 until the ordering issue is resolved.")

print("=" * 80)

# Close DuckDB connection
con.close()

PHASE 2.10.3 — FINAL CHRONOLOGICAL RE-SORT & VERIFICATION

Input directory : /content/phase2_numeric_cleaned
Output directory: /content/phase2_final
Files found     : 10

--------------------------------------------------------------------------------
[1/10] Friday-02-03-2018_TrafficForML_CICFlowMeter.csv
--------------------------------------------------------------------------------


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Rows                 : 1,043,116
Minimum Timestamp    : 2018-03-02 01:00:00
Maximum Timestamp    : 2018-03-02 12:59:59
Invalid Timestamps   : 0
Ordering Violations  : 0
Chronological        : True

--------------------------------------------------------------------------------
[2/10] Friday-16-02-2018_TrafficForML_CICFlowMeter.csv
--------------------------------------------------------------------------------


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Rows                 : 900,988
Minimum Timestamp    : 2018-02-16 01:00:32
Maximum Timestamp    : 2018-02-16 12:58:24
Invalid Timestamps   : 0
Ordering Violations  : 0
Chronological        : True

--------------------------------------------------------------------------------
[3/10] Friday-23-02-2018_TrafficForML_CICFlowMeter.csv
--------------------------------------------------------------------------------


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

... (13 outputs left out)

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Rows                 : 1,031,018
Minimum Timestamp    : 2018-02-21 01:55:46
Maximum Timestamp    : 2018-02-21 10:43:21
Invalid Timestamps   : 0
Ordering Violations  : 0
Chronological        : True

--------------------------------------------------------------------------------
[10/10] Wednesday-28-02-2018_TrafficForML_CICFlowMeter.csv
--------------------------------------------------------------------------------


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Rows                 : 606,982
Minimum Timestamp    : 2018-02-28 01:00:00
Maximum Timestamp    : 2018-02-28 12:59:59
Invalid Timestamps   : 0
Ordering Violations  : 0
Chronological        : True

FINAL CHRONOLOGICAL SORT SUMMARY
                                              File    Rows   Minimum Timestamp   Maximum Timestamp  Invalid Timestamps  Ordering Violations  Chronological
   Friday-02-03-2018_TrafficForML_CICFlowMeter.csv 1043116 2018-03-02 01:00:00 2018-03-02 12:59:59                   0                    0           True
   Friday-16-02-2018_TrafficForML_CICFlowMeter.csv  900988 2018-02-16 01:00:32 2018-02-16 12:58:24                   0                    0           True
   Friday-23-02-2018_TrafficForML_CICFlowMeter.csv 1045960 2018-02-23 01:00:00 2018-02-23 12:59:59                   0                    0           True
 Thuesday-20-02-2018_TrafficForML_CICFlowMeter.csv 7948746 2018-02-20 01:00:00 2018-02-20 12:59:59                   0                    0           T

#All 10 CIC-IDS2018 files have now been successfully re-sorted and independently verified chronologically. The final dataset contains 15,822,221 rows, with 0 invalid timestamps and 0 ordering violations across all files.

The final chronological verification reports True for every file. The final directory is:

/content/phase2_final